# The comparator with three numbers

`max(n1, n2, n3)` on Qwen2.5-7B-Instruct, one-shot prompt. Everything is read at the **third**
number position $n_3$, where all three numbers are present:

$$\mathbf{v}_1 = \text{H14 DAS 1D},\qquad
  \mathbf{v}_2 = \text{H18 DAS 1D},\qquad
  \mathbf{v}_3 = \text{L13 PC1 @ } n_3$$

$\mathbf{v}_1,\mathbf{v}_2$ are copied into $n_3$ by layer-14 heads; $\mathbf{v}_3$ is already
there. Three perturbation cases, one per number: `y1` ($s_1>s_3>s_2$), `y2` ($s_2>s_3>s_1$),
`y3` ($s_3>s_2>s_1$).

Every block caches its `.npz` and draws its figure immediately. See
`docs/explore_multicompare.md`.

In [ ]:
import os, json, time, importlib
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  (registers the 3d projection)

import data_numberreps as D
import data_sharedrep as S
import data_comparator as C
import data_multicompare as MC
importlib.reload(MC)          # pick up edits to the module without restarting the kernel
from data_multicompare import (CASES, CASE_LIST, HEAD_GROUPS, ALL_ORDERS, LAYER, H_LAYER,
                               HEAD_V1, HEAD_V2, REGIMES, N_EVAL, N_TRAIN, N_CLOUD,
                               order_label, ck)
from data_numberreps import SEED, EVAL_BS, device

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

REGIME = '2digit'
LOW, HIGH, GAP = (REGIMES[REGIME][k] for k in ('low', 'high', 'gap'))

NEURON_LAYERS = [14, 15]
D_FF = 18944                    # Qwen2.5-7B intermediate size; asserted against the model below
POOL_LAYER  = np.concatenate([np.full(D_FF, l) for l in NEURON_LAYERS])
POOL_WITHIN = np.concatenate([np.arange(D_FF) for _ in NEURON_LAYERS])
NAME = lambda p: f'L{POOL_LAYER[p]}#{POOL_WITHIN[p]}'

# The two-number results, for the last section (how the neurons found there behave here).
TWO_PATH = f'{RESULTS_DIR}/comparator_data_{REGIME}.npz'
TWO = dict(np.load(TWO_PATH)) if os.path.exists(TWO_PATH) else None
print('two-number results:', 'loaded' if TWO else f'MISSING ({TWO_PATH})')
if TWO is not None:
    print(f"  overlap_k20 = {len(TWO['overlap_k20'])} neurons: "
          + ', '.join(NAME(p) for p in TWO['overlap_k20']))


def ensure_model():
    # Load the model on demand for a self-contained cell -- independent of PLOT_ONLY, which is
    # computed once from the *original* BLOCKS list and goes stale the moment a self-contained
    # block (added after that) turns out to be missing in a session where everything else is
    # already cached.
    if D.model is None:
        D.load_model()
        S.init_head_machinery()

## Plot style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 16, 13, 13, 17
DPI, SAVE = 300, True

# Same palette as makefig_sharedreps / explore_comparatorfig, extended with a third number.
COL = {'full': '#2166AC',   # blue
       'das':  '#C97B00',   # amber
       'pc1':  '#C06A9B',   # orchid
       'h14':  '#117777',   # teal
       'ceil': '#6E6E6E'}   # grey

# One colour per number, used for the case it perturbs and for the direction that carries it.
NCOL = {1: '#C97B00', 2: '#2166AC', 3: '#2E8B57'}
VCOL = {1: '#117777', 2: '#C06A9B', 3: '#6A51A3'}
CASE_COL = {'y1': NCOL[1], 'y2': NCOL[2], 'y3': NCOL[3]}
GROUP_COL = {'s1>s2': '#6A51A3', 's1>s3': NCOL[1], 's2>s3': NCOL[2]}
JOINT_HATCH = '//'

V = {1: r'$\mathbf{v}_1$', 2: r'$\mathbf{v}_2$', 3: r'$\mathbf{v}_3$'}
Y = {1: r'$y_1$', 2: r'$y_2$', 3: r'$y_3$'}
CMAP = 'viridis'


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/multicompare_{name}.png', dpi=DPI, bbox_inches='tight')


def aggregate(x, y):
    '''Per-distinct-x mean and standard error of y.'''
    u = np.unique(x)
    m = np.array([y[x == v].mean() for v in u])
    se = np.array([y[x == v].std() / max(np.sqrt((x == v).sum()), 1) for v in u])
    return u, m, se


def barlab(ax, xpos, mean, se, text, fs=10):
    ax.text(xpos, mean + se + 0.02, text, ha='center', va='bottom', fontsize=fs)

## Run mode

Each block writes one `.npz`. When all of them exist the model is never loaded and the notebook
runs as a pure plotting pass.

In [ ]:
BLOCKS = ['pool', 'heads', 'dirs', 'space', 'sweep', 'freeze', 'attr', 'rf', 'connect',
          'transfer', 'das15', 'ucos', 'heads_alone', 'late', 'latehead']
PATH = {b: f'{RESULTS_DIR}/multicompare_{b}_{REGIME}.npz' for b in BLOCKS}

FORCE = set()        # block names to recompute even though their file exists, e.g. {'sweep'}

_missing = [b for b in BLOCKS if not os.path.exists(PATH[b])]
PLOT_ONLY = not _missing and not FORCE
print('run mode:', 'PLOT ONLY -- every result is cached, the model is not loaded' if PLOT_ONLY
      else f'COMPUTE -- missing: {_missing or "none"}   forced: {sorted(FORCE) or "none"}')


def cached(block, compute):
    '''Load `PATH[block]` if present, else run `compute()` -> {name: array}, save it, return it.'''
    p = PATH[block]
    if os.path.exists(p) and block not in FORCE:
        d = dict(np.load(p, allow_pickle=True))
        print(f'loaded {p}   ({len(d)} arrays)')
        return d
    assert not PLOT_ONLY, f'{p} is not cached -- set PLOT_ONLY = False and run the model cells'
    t = time.time()
    d = compute()
    np.savez_compressed(p, **{k: np.asarray(v) for k, v in d.items()})
    print(f'computed and saved {p}   ({len(d)} arrays, {time.time() - t:.0f} s)')
    return d

## Model and token layout

Separate cell — everything above runs without it.

In [ ]:
if PLOT_ONLY:
    model = tokenizer = None
    print('plot-only: model not loaded')
else:
    model, tokenizer = D.load_model()
    S.init_head_machinery()
    assert D.model.config.intermediate_size == D_FF
    assert S.n_heads == D.d_model // D.head_dim

T, POS, NTOK, REF = (None,) * 4
if not PLOT_ONLY:
    T, POS, NTOK, REF = MC.regime_positions(LOW, HIGH)
    print(f'T={T}   number last-tokens {POS}   tokens per number {NTOK}')
    print(f'reference prompt: {REF!r}')
POS3_KEY = 'pos3'

## The quadruple pool

Each quadruple $v_0>v_1>v_2>v_3$ supplies three operands plus the replacement that corrupts the
winner. Kept only if the model solves it — clean run answers $v_0$, corrupted run answers $v_1$ —
in **every** value order used below, so all orders are measured on the same values.

In [ ]:
N_POOL = 1400            # drawn before filtering; the assert below says if it was not enough
N_KEEP = N_EVAL + N_TRAIN


def _pool():
    quads = MC.build_quads(N_POOL, SEED, LOW, HIGH, GAP)
    ok = np.ones(len(quads), dtype=bool)
    per = {}
    for o in ALL_ORDERS:
        m = MC.solved_mask(MC.make_batch(quads, o, T))
        per[order_label(o)] = m.mean()
        ok &= m
        print(f'  {order_label(o)}: solved {m.mean():.3f}', flush=True)
    idx = np.where(ok)[0]
    print(f'  solved in all {len(ALL_ORDERS)} orders: {len(idx)}/{len(quads)} '
          f'({len(idx) / len(quads):.3f})', flush=True)
    assert len(idx) >= N_KEEP, f'only {len(idx)} usable quadruples; raise N_POOL'
    kept = np.array([quads[i] for i in idx[:N_KEEP]], dtype=np.int64)
    return dict(quads=kept, eval_quads=kept[:N_EVAL], train_quads=kept[N_EVAL:],
                solved_frac_all=np.array(len(idx) / len(quads)),
                solved_frac=np.array([per[order_label(o)] for o in ALL_ORDERS]),
                orders=np.array(ALL_ORDERS), T=np.array(T if T else 0),
                pos=np.array(POS if POS else [0, 0, 0]))


POOLD = cached('pool', _pool)
EVAL_Q = [tuple(q) for q in POOLD['eval_quads']]
TRAIN_Q = [tuple(q) for q in POOLD['train_quads']]
if PLOT_ONLY:
    T, POS = int(POOLD['T']), POOLD['pos'].tolist()
POS3 = POS[2]
print(f'{len(EVAL_Q)} eval + {len(TRAIN_Q)} train quadruples   n3 token {POS3}   '
      f'per-order solve rate ' + ', '.join(f'{order_label(o)} {f:.2f}'
                                           for o, f in zip(ALL_ORDERS, POOLD['solved_frac'])))

## Batches and clean-run sources

One batch per value order, built once and reused by every block.

In [ ]:
BAT, SRC = {}, {}


def order_pos(o, at_n3=True):
    '''Patch site for an order: n3 for the shared-representation work, the runner-up for the
    per-head sweep.'''
    return POS[2] if at_n3 else POS[o[1]]


def get_bat(o):
    if o not in BAT:
        b = MC.make_batch(EVAL_Q, o, T)
        b['pld_clean'], b['pld_corr'] = MC.pos_anchors(b)
        BAT[o] = b
    return BAT[o]


def get_src(o, pos, layers=(LAYER, H_LAYER)):
    key = (o, pos, tuple(layers))
    if key not in SRC:
        SRC[key] = MC.sources_for(get_bat(o), pos, layers)
    return SRC[key]


if not PLOT_ONLY:
    for o in ALL_ORDERS:
        b = get_bat(o)
        print(f'{order_label(o)}: clean PLD {b["pld_clean"].mean():+6.2f}   '
              f'corrupted PLD {b["pld_corr"].mean():+6.2f}   '
              f'gap {(b["pld_clean"] - b["pld_corr"]).mean():5.2f}', flush=True)

# 1. Which L14 head moves which number

Each L14 head's own output contribution interchanged at the group's runner-up position, co-patched
with the full clean L13 residual there. Three groups, named by the **top two** slots of the clean
value order.

| group | clean order | patch site |
|---|---|---|
| `s1>s2` | $s_1>s_2>s_3$ | $n_2$ |
| `s1>s3` | $s_1>s_3>s_2$ | $n_3$ |
| `s2>s3` | $s_2>s_3>s_1$ | $n_3$ |

In [ ]:
def _heads():
    out = {}
    for gname, o in HEAD_GROUPS.items():
        pos = order_pos(o, at_n3=False)
        bat, src = get_bat(o), get_src(o, pos, layers=(LAYER, H_LAYER))
        base = dict(resid=[(LAYER, 'full', None)])
        for nm, spec in (('L13', base),
                         ('L14', dict(resid=[(H_LAYER, 'full', None)]))):
            rec, iia = MC.eval_spec(spec, bat, src, pos)
            out[f'base_{ck(gname)}_{nm}_posrec'] = rec.astype(np.float32)
            out[f'base_{ck(gname)}_{nm}_iia'] = iia.astype(np.float32)
        pr = np.zeros((S.n_heads, bat['n']), dtype=np.float32)
        ii = np.zeros((S.n_heads, bat['n']), dtype=np.float32)
        for h in range(S.n_heads):
            spec = dict(resid=[(LAYER, 'full', None)], heads=[(h, None)])
            pr[h], ii[h] = MC.eval_spec(spec, bat, src, pos)
        out[f'posrec_{ck(gname)}'] = pr
        out[f'iia_{ck(gname)}'] = ii
        out[f'pos_{ck(gname)}'] = np.array(pos)
        top = np.argsort(-pr.mean(1))[:5]
        print(f'{gname:>6} @tok{pos}  L13 {out[f"base_{ck(gname)}_L13_posrec"].mean():+.3f}  '
              f'L14 {out[f"base_{ck(gname)}_L14_posrec"].mean():+.3f}  top: '
              + ', '.join(f'H{h}({pr[h].mean():+.2f})' for h in top), flush=True)
    out['n_heads'] = np.array(S.n_heads)
    return out


HEADS = cached('heads', _heads)
N_HEADS = int(HEADS['n_heads'])

### Figure — three bars per head

In [ ]:
RANK_GROUP = 's1>s3'       # x-axis ordering; the group v1 is defined in
SHOW_HEADS = None          # e.g. 12 to crop to the top ones; None = all

order_ix = np.argsort(-HEADS[f'posrec_{ck(RANK_GROUP)}'].mean(1))
if SHOW_HEADS:
    order_ix = order_ix[:SHOW_HEADS]
x, w = np.arange(len(order_ix)), 0.8 / len(HEAD_GROUPS)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
for ax, (key, ylab) in zip(axes, [('posrec', 'position recovery'), ('iia', 'IIA')]):
    for i, g in enumerate(HEAD_GROUPS):
        v = HEADS[f'{key}_{ck(g)}'][order_ix]
        ax.bar(x + (i - (len(HEAD_GROUPS) - 1) / 2) * w, v.mean(1), w,
               yerr=v.std(1) / np.sqrt(v.shape[1]), color=GROUP_COL[g],
               error_kw=dict(lw=0.7, ecolor='0.3'), alpha=0.92,
               label=f'{g}   @ $n_{{{HEAD_GROUPS[g][1] + 1}}}$')
        ax.axhline(HEADS[f'base_{ck(g)}_L13_{key}'].mean(), ls='--', lw=1.1,
                   color=GROUP_COL[g], alpha=0.75)
    ax.axhline(0, color='0.6', lw=0.9)
    style_axes(ax, ylabel=ylab)
axes[0].legend(fontsize=LEGEND_FS, ncol=3, loc='upper right',
               title='top-two order (dashed: its L13-alone baseline)',
               title_fontsize=LEGEND_FS)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'H{h}' for h in order_ix], rotation=90, fontsize=10)
axes[1].set_xlabel(f'layer {H_LAYER} attention head ({RANK_GROUP} ranking)', fontsize=LABEL_FS)
fig.suptitle('single-head interchange at the runner-up position, co-patched with L13',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'L14_head_orders'); plt.show()

for g in HEAD_GROUPS:
    pr = HEADS[f'posrec_{ck(g)}'].mean(1)
    print(f'{g:>6}: ' + '  '.join(f'H{h} {pr[h]:+.3f}' for h in np.argsort(-pr)[:6]))

### The same sweep with no L13 co-patch

Each head's own output contribution interchanged at the group's runner-up position and **nothing
else**. The co-patched version above asks what a head adds on top of a restored L13 residual; this
asks what it does by itself. Dashed line is the full L14 residual patch at the same position — the
ceiling for everything layer-14 attention can do without help.

In [ ]:
def _heads_alone():
    out = {}
    for gname, o in HEAD_GROUPS.items():
        pos = order_pos(o, at_n3=False)
        # Same sources key as the co-patched sweep, so nothing is captured twice in one session.
        bat, src = get_bat(o), get_src(o, pos, layers=(LAYER, H_LAYER))
        pr = np.zeros((S.n_heads, bat['n']), dtype=np.float32)
        ii = np.zeros((S.n_heads, bat['n']), dtype=np.float32)
        for h in range(S.n_heads):
            pr[h], ii[h] = MC.eval_spec(dict(heads=[(h, None)]), bat, src, pos)
        out[f'posrec_{ck(gname)}'] = pr
        out[f'iia_{ck(gname)}'] = ii
        top = np.argsort(-pr.mean(1))[:5]
        print(f'{gname:>6} @tok{pos}  alone  top: '
              + ', '.join(f'H{h}({pr[h].mean():+.2f})' for h in top), flush=True)
    return out


HEADS_ALONE = cached('heads_alone', _heads_alone)

In [ ]:
# Same ranking and colours as the co-patched figure, so the two read as one pair.
order_a = np.argsort(-HEADS_ALONE[f'posrec_{ck(RANK_GROUP)}'].mean(1))
if SHOW_HEADS:
    order_a = order_a[:SHOW_HEADS]
xa, wa = np.arange(len(order_a)), 0.8 / len(HEAD_GROUPS)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
for ax, (key, ylab) in zip(axes, [('posrec', 'position recovery'), ('iia', 'IIA')]):
    for i, g in enumerate(HEAD_GROUPS):
        v = HEADS_ALONE[f'{key}_{ck(g)}'][order_a]
        ax.bar(xa + (i - (len(HEAD_GROUPS) - 1) / 2) * wa, v.mean(1), wa,
               yerr=v.std(1) / np.sqrt(v.shape[1]), color=GROUP_COL[g],
               error_kw=dict(lw=0.7, ecolor='0.3'), alpha=0.92,
               label=f'{g}   @ $n_{{{HEAD_GROUPS[g][1] + 1}}}$')
        # reference: the whole L14 residual patched at the same position, also without L13
        ax.axhline(HEADS[f'base_{ck(g)}_L14_{key}'].mean(), ls='--', lw=1.1,
                   color=GROUP_COL[g], alpha=0.75)
    ax.axhline(0, color='0.6', lw=0.9)
    style_axes(ax, ylabel=ylab)
axes[0].legend(fontsize=LEGEND_FS, ncol=3, loc='upper right',
               title='top-two order (dashed: its full-L14-residual patch)',
               title_fontsize=LEGEND_FS)
axes[1].set_xticks(xa)
axes[1].set_xticklabels([f'H{h}' for h in order_a], rotation=90, fontsize=10)
axes[1].set_xlabel(f'layer {H_LAYER} attention head ({RANK_GROUP} ranking, head alone)',
                   fontsize=LABEL_FS)
fig.suptitle('single-head interchange at the runner-up position, no L13 co-patch',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'L14_head_orders_alone'); plt.show()

# What the L13 co-patch adds, for the heads that matter in either version.
print(f'{"head":>6}' + ''.join(f'{g + " alone":>14}{g + " +L13":>13}' for g in HEAD_GROUPS))
_top = sorted({int(h) for g in HEAD_GROUPS
               for h in np.argsort(-HEADS_ALONE[f'posrec_{ck(g)}'].mean(1))[:3]}
              | {int(h) for g in HEAD_GROUPS
                 for h in np.argsort(-HEADS[f'posrec_{ck(g)}'].mean(1))[:3]})
for h in _top:
    print(f'H{h:<5}' + ''.join(f'{HEADS_ALONE[f"posrec_{ck(g)}"][h].mean():>14.3f}'
                               f'{HEADS[f"posrec_{ck(g)}"][h].mean():>13.3f}'
                               for g in HEAD_GROUPS))

# 2. The three directions at $n_3$

$\mathbf{v}_1$, $\mathbf{v}_2$: rank-1 DAS inside H14's and H18's own output contribution, fitted
on the `y1` / `y2` cases. $\mathbf{v}_3$: PC1 of the L13 residual cloud at $n_3$ — not fitted.
Signs are oriented so each rises with its own number.

In [ ]:
DAS_SEEDS = [0]          # add seeds to check stability; the first is the one used downstream


def _dirs():
    out = {}
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    cloud = MC.capture_cloud(trips, T, POS3)
    yv = np.array(trips, dtype=np.int64)
    out['cloud_y'] = yv
    for k, v in cloud.items():
        out[f'cloud_{k}'] = v

    # v3: PC1 of the L13 residual at n3 (three-component PCA kept for the scatter panels).
    v3, pca13 = MC.pc1_of(cloud['l13'])
    out['pca13_components'] = pca13.components_.astype(np.float32)
    out['pca13_evr'] = pca13.explained_variance_ratio_.astype(np.float32)

    # v1, v2: rank-1 DAS in each head's output, each trained on the case whose number it carries.
    dirs = {}
    for j, (head, case) in enumerate([(HEAD_V1, 'y1'), (HEAD_V2, 'y2')], start=1):
        o = CASES[case]
        tbat = MC.make_batch(TRAIN_Q, o, T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        ctx = S.capture_ctx(tbat['ids_c'], POS3)
        for s in DAS_SEEDS:
            Q, curve = MC.train_head_das(head, tbat, ctx, POS3, seed=s)
            out[f'das_curve_H{head}_s{s}'] = curve
            if s == DAS_SEEDS[0]:
                dirs[j] = Q[:, 0].cpu().numpy()
            else:
                out[f'das_dir_H{head}_s{s}'] = Q[:, 0].cpu().numpy().astype(np.float32)
            print(f'  H{head} DAS seed {s}: loss {curve[0]:.3f} -> {curve[-1]:.3f}', flush=True)
        del ctx, tbat
        torch.cuda.empty_cache()
    dirs[3] = v3

    # Orient every direction so its component rises with its own number, then store.
    for j in (1, 2, 3):
        d = dirs[j] / np.linalg.norm(dirs[j])
        src = cloud[f'head{HEAD_V1 if j == 1 else HEAD_V2}'] if j < 3 else cloud['l13']
        r = np.corrcoef(src @ d, np.log(yv[:, j - 1]))[0, 1]
        out[f'v{j}'] = (d * np.sign(r)).astype(np.float32)
        out[f'v{j}_corr'] = np.array(abs(r))
        print(f'  v{j}: |corr| with log y{j} on its own cloud = {abs(r):.3f}', flush=True)

    # Probes: ridge log-magnitude readouts on the pre-MLP L14 cloud, the comparator's input.
    probes = MC.fit_probes(cloud['pre14'], {f'p{j}': yv[:, j - 1].astype(float) for j in (1, 2, 3)})
    for nm, (w, r2) in probes.items():
        out[nm], out[f'{nm}_r2'] = w, np.array(r2)
        print(f'  probe {nm}: held-out R2 {r2:.3f}', flush=True)
    return out


DIRS = cached('dirs', _dirs)
CL = {k[6:]: DIRS[k] for k in DIRS if k.startswith('cloud_') and k != 'cloud_y'}
YV = DIRS['cloud_y']
VDIR = {j: DIRS[f'v{j}'].astype(np.float64) for j in (1, 2, 3)}
PROBE = {j: DIRS[f'p{j}'].astype(np.float64) for j in (1, 2, 3)}

# The patch / display basis: Gram-Schmidt on (v1, v2, v3). Axis j is vj orthogonalised against the
# ones before it, so axis 1 IS v1 and the frame is the three-number version of the (v1, v2) plane.
_Vst = np.stack([VDIR[1], VDIR[2], VDIR[3]], axis=1)
_Q, _R = np.linalg.qr(_Vst)
PB = (_Q * np.sign(np.diag(_R))).astype(np.float32)          # (d_model, 3), orthonormal
print('basis check  diag(P^T V) =', np.round(np.diag(PB.T.astype(np.float64) @ _Vst), 3))

### Figure — IIA of each direction, by which number is perturbed

In [ ]:
def Pvec(j):
    '''Direction v_j as a (d_model, 1) orthonormal patch basis.'''
    return torch.tensor(VDIR[j].astype(np.float32), device=device).reshape(-1, 1)


def _iia_block():
    '''Each direction against the full-rank interchange at its own site, in every case.'''
    out = {}
    ARMS = {'L13 full @n3': dict(resid=[(LAYER, 'full', None)]),
            'v3 (L13 PC1)': dict(resid=[(LAYER, 'sub', Pvec(3))]),
            'H14 full out': dict(heads=[(HEAD_V1, None)]),
            'v1 (H14 DAS)': dict(heads=[(HEAD_V1, Pvec(1))]),
            'H18 full out': dict(heads=[(HEAD_V2, None)]),
            'v2 (H18 DAS)': dict(heads=[(HEAD_V2, Pvec(2))])}
    for case in CASE_LIST:
        o = CASES[case]
        bat, src = get_bat(o), get_src(o, POS3)
        for nm, spec in ARMS.items():
            rec, iia = MC.eval_spec(spec, bat, src, POS3)
            out[f'posrec_{case}_{ck(nm)}'] = rec.astype(np.float32)
            out[f'iia_{case}_{ck(nm)}'] = iia.astype(np.float32)
        print(f'{case}: ' + '  '.join(f'{nm} {out[f"iia_{case}_{ck(nm)}"].mean():.2f}'
                                      for nm in ARMS), flush=True)
    out['arm_names'] = np.array(list(ARMS))
    return out

In [ ]:
# Drawn from the `space` block below, which runs this together with the 3-D conditions.
IIA_ARMS = [('L13 full @n3', 'full', COL['full']),
            ('v1 (H14 DAS)', V[1],   VCOL[1]),
            ('v2 (H18 DAS)', V[2],   VCOL[2]),
            ('v3 (L13 PC1)', V[3],   VCOL[3])]
HEAD_FULL = {'y1': ('H14 full out', 'H14 full'), 'y2': ('H18 full out', 'H18 full')}


def draw_iia_by_number(Z, fname):
    '''One group per number; inside a group, the full-scope control and that number's direction.'''
    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    W = 0.26
    for gi, case in enumerate(CASE_LIST):
        j = int(case[1])
        bars = [('L13 full @n3', 'L13 full', COL['full']),
                (IIA_ARMS[j][0], IIA_ARMS[j][1], VCOL[j])]
        if case in HEAD_FULL:
            bars.insert(1, (HEAD_FULL[case][0], HEAD_FULL[case][1], COL['ceil']))
        for bi, (key, lab, col) in enumerate(bars):
            v = Z[f'iia_{case}_{ck(key)}']
            xp = gi + (bi - (len(bars) - 1) / 2) * W
            se = v.std() / np.sqrt(len(v))
            ax.bar(xp, v.mean(), W, yerr=se, capsize=4, color=col, alpha=0.92)
            barlab(ax, xp, v.mean(), se, f'{lab}\n{v.mean():.2f}')
    ax.set_xticks(range(3))
    ax.set_xticklabels([f'{Y[j]} perturbed' for j in (1, 2, 3)], fontsize=LABEL_FS)
    ax.set_ylim(0, 1.25)
    style_axes(ax, ylabel='IIA')
    plt.tight_layout(); savefig(fig, fname); plt.show()

# 3. The shared representation in 3D

Pre-MLP L14 residual at $n_3$, projected on the orthonormalised $(\mathbf{v}_1,\mathbf{v}_2,
\mathbf{v}_3)$ frame, one panel per colouring.

In [ ]:
PQ = (CL['pre14'].astype(np.float64) - CL['pre14'].astype(np.float64).mean(0)) @ PB
ELEV, AZIM = 22, -58

fig = plt.figure(figsize=(17.5, 6.6))
for j in (1, 2, 3):
    ax = fig.add_subplot(1, 3, j, projection='3d')
    sc = ax.scatter(PQ[:, 0], PQ[:, 1], PQ[:, 2], c=YV[:, j - 1], cmap=CMAP, s=7, alpha=0.6,
                    edgecolors='none')
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_box_aspect((1, 1, 0.92))
    ax.set_xlabel(V[1], fontsize=LABEL_FS, labelpad=6)
    ax.set_ylabel(V[2], fontsize=LABEL_FS, labelpad=6)
    ax.set_zlabel(V[3], fontsize=LABEL_FS, labelpad=4)
    ax.tick_params(labelsize=9, pad=0.5)
    cb = fig.colorbar(sc, ax=ax, shrink=0.55, pad=0.10, aspect=18)
    cb.set_label(Y[j], fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
    cb.outline.set_visible(False)
fig.suptitle('pre-MLP L14 residual at $n_3$ in the $(v_1, v_2, v_3)$ frame', fontsize=TITLE_FS)
# tight_layout leaves a 3d axes' unused corners as figure margin; set the box directly instead.
fig.subplots_adjust(left=0.005, right=0.985, top=0.99, bottom=0.0, wspace=0.10)
savefig(fig, 'space_3d'); plt.show()

for j in (1, 2, 3):
    r = [np.corrcoef(PQ[:, a], np.log(YV[:, j - 1]))[0, 1] for a in range(3)]
    print(f'log y{j} vs axes: ' + '  '.join(f'v{a + 1} {r[a]:+.3f}' for a in range(3)))

### Patching the space

Each direction alone, the rank-3 span, and the two full-rank ceilings at the same position.

In [ ]:
def _space():
    out = _iia_block()
    Pt = torch.tensor(PB, device=device)

    ARMS = {'v1 only':      dict(heads=[(HEAD_V1, Pvec(1))]),
            'v2 only':      dict(heads=[(HEAD_V2, Pvec(2))]),
            'v3 only':      dict(resid=[(LAYER, 'sub', Pvec(3))]),
            'v1+v2+v3 sep': dict(heads=[(HEAD_V1, Pvec(1)), (HEAD_V2, Pvec(2))],
                                 resid=[(LAYER, 'sub', Pvec(3))]),
            'span preMLP':  dict(premlp=Pt),
            'span postMLP': dict(resid=[(H_LAYER, 'sub', Pt)]),
            'L13 full':     dict(resid=[(LAYER, 'full', None)]),
            'L14 full':     dict(resid=[(H_LAYER, 'full', None)]),
            'L14 full preMLP': dict(premlp=None)}
    for case in CASE_LIST:
        o = CASES[case]
        bat, src = get_bat(o), get_src(o, POS3)
        for nm, spec in ARMS.items():
            rec, iia = MC.eval_spec(spec, bat, src, POS3)
            out[f'sp_posrec_{case}_{ck(nm)}'] = rec.astype(np.float32)
            out[f'sp_iia_{case}_{ck(nm)}'] = iia.astype(np.float32)
        print(f'{case}: ' + '  '.join(f'{nm} {out[f"sp_iia_{case}_{ck(nm)}"].mean():.2f}'
                                      for nm in ARMS), flush=True)
    out['sp_arm_names'] = np.array(list(ARMS))
    out['plane_basis'] = PB
    out['plane_pq'] = PQ.astype(np.float32)
    return out


SPACE = cached('space', _space)

In [ ]:
draw_iia_by_number(SPACE, 'iia_by_number')

In [ ]:
SP_COND = [('v3 only',      V[3],                       False),
           ('v2 only',      V[2],                       False),
           ('v1 only',      V[1],                       False),
           ('v1+v2+v3 sep', f'{V[1]},{V[2]},{V[3]}\nseparate', True),
           ('span preMLP',  'span\npre-MLP',            True),
           ('span postMLP', 'span\npost-MLP',           True),
           ('L13 full',     'L13\nfull',                False),
           ('L14 full preMLP', 'L14 full\npre-MLP',     False),
           ('L14 full',     'L14\nfull',                False)]

fig, axes = plt.subplots(1, 2, figsize=(19, 5.8))
for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.28)),
                                        ('posrec', 'position recovery', None)]):
    xs, W = np.arange(len(SP_COND)), 0.8 / len(CASE_LIST)
    for ci, case in enumerate(CASE_LIST):
        j = int(case[1])
        for k, (nm, _l, hatch) in enumerate(SP_COND):
            v = SPACE[f'sp_{key}_{case}_{ck(nm)}']
            xp = xs[k] + (ci - (len(CASE_LIST) - 1) / 2) * W
            se = v.std() / np.sqrt(len(v))
            ax.bar(xp, v.mean(), W, yerr=se, capsize=2.5, color=CASE_COL[case], alpha=0.92,
                   hatch=JOINT_HATCH if hatch else '', edgecolor='white', linewidth=0,
                   label=f'{Y[j]} perturbed' if k == 0 else None)
    ax.set_xticks(xs); ax.set_xticklabels([c[1] for c in SP_COND], fontsize=11)
    ax.axhline(0, color='0.6', lw=0.9)
    if ylim: ax.set_ylim(*ylim)
    style_axes(ax, ylabel=ylab, xlabel='patched component')
    ax.legend(fontsize=LEGEND_FS, loc='upper left', framealpha=0.92)
fig.suptitle('interchange at $n_3$: each direction, the rank-3 span, and the full-rank ceilings',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'space_patching'); plt.show()

for nm, _l, _h in SP_COND:
    print(f'{nm:>18}  ' + '  '.join(f'{c} {SPACE[f"sp_iia_{c}_{ck(nm)}"].mean():.3f}'
                                    for c in CASE_LIST))

# 4. Angles between the directions, and the probes

Cosine similarity of $\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3$ and the three ridge log-magnitude
probes $\mathbf{p}_1,\mathbf{p}_2,\mathbf{p}_3$ fitted on the same cloud. Chance is
$1/\sqrt{d_\text{model}}$.

In [ ]:
from matplotlib.patches import Rectangle

NAMES = [('v', 1), ('v', 2), ('v', 3), ('p', 1), ('p', 2), ('p', 3)]
VECS = [VDIR[j] if t == 'v' else PROBE[j] for t, j in NAMES]
LBL = [V[j] if t == 'v' else rf'$\mathbf{{p}}_{j}$' for t, j in NAMES]
Cm = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) for b in VECS]
               for a in VECS])
d_model = len(VDIR[1])

fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.2), width_ratios=[1, 1.35])
ax = axes[0]
im = ax.imshow(np.where(np.tril(np.ones_like(Cm)) > 0, Cm, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(6)); ax.set_xticklabels(LBL, fontsize=15)
ax.set_yticks(range(6)); ax.set_yticklabels(LBL, fontsize=15)
for i in range(6):
    for j in range(i + 1):
        ax.text(j, i, f'{Cm[i, j]:+.2f}', ha='center', va='center', fontsize=12,
                color='white' if abs(Cm[i, j]) > 0.6 else 'black')
for i, j in [(1, 0), (2, 0), (2, 1)]:
    ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor='#111111', lw=2.6))
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
cb.set_label('cosine similarity', fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values(): s_.set_visible(False)
ax.set_title(f'directions (boxed: the three causal pairs)', fontsize=TITLE_FS)

ax = axes[1]
for j in (1, 2, 3):
    comp = CL['pre14'].astype(np.float64) @ PROBE[j]
    xg, m, se = aggregate(YV[:, j - 1], comp - comp.mean())
    ax.errorbar(xg, m, yerr=se, fmt='o', ms=4.5, color=NCOL[j], linestyle='none',
                elinewidth=1.1, capsize=2, alpha=0.9,
                label=rf'$\mathbf{{p}}_{j}$ ($R^2$={float(DIRS[f"p{j}_r2"]):.2f})')
ax.axhline(0, color='0.7', lw=0.9)
style_axes(ax, xlabel='number value at that slot', ylabel='component along the probe', grid=False)
ax.grid(alpha=0.3)
ax.legend(fontsize=LEGEND_FS)
ax.set_title('pre-MLP L14 residual @ $n_3$, read by each probe', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'angles_probes'); plt.show()

ang = lambda i, j: np.degrees(np.arccos(np.clip(Cm[i, j], -1, 1)))
PLAIN = [f'{t}{j}' for t, j in NAMES]
for i, j in [(1, 0), (2, 0), (2, 1), (4, 3), (5, 3), (5, 4)]:
    print(f'{PLAIN[i]:>3} vs {PLAIN[j]:>3}: cos {Cm[i, j]:+.3f}  ({ang(i, j):5.1f} deg)')
print(f'chance |cos| = 1/sqrt(d_model) = {1 / np.sqrt(d_model):.4f}')
print('probe held-out R2: ' + '  '.join(f'p{j} {float(DIRS[f"p{j}_r2"]):.3f}' for j in (1, 2, 3)))

# 5. Probability contours over the space

The three coordinates are **set** to the cloud-mean coordinate of a chosen $(y_1^*, y_2^*, y_3^*)$
over a $9^3$ grid, and the answered slot is recorded:

$$P_j(y_1^*,y_2^*,y_3^*) = \Pr[\text{the model names its own slot-}j\text{ number}]$$

In [ ]:
SWEEP_CASE = 'y1'
SWEEP_N = 24          # prompts per grid cell


def _sweep():
    grid = MC.sweep_grid_values(LOW, HIGH)
    o = CASES[SWEEP_CASE]
    bat = MC._slice_batch(get_bat(o), 0, SWEEP_N)
    coords = CL['pre14'].astype(np.float64) @ PB          # raw (uncentred) frame the patch uses
    coord_of = MC.mean_coords(coords, YV, grid)
    am = MC.sweep_space(bat, torch.tensor(PB, device=device), coord_of, grid, POS3)
    P = np.stack([(am == bat['tok_slot'][:, j].cpu().numpy()[None, None, None, :]).mean(-1)
                  for j in range(3)])                     # (3, G, G, G)
    named = np.stack([(am == bat['tok_slot'][:, j].cpu().numpy()[None, None, None, :])
                      for j in range(3)]).any(0).mean(-1)
    return dict(grid=np.array(grid), P=P.astype(np.float32), named=named.astype(np.float32),
                coord_of=np.stack([coord_of[y] for y in grid]).astype(np.float32),
                case=np.array(SWEEP_CASE), n_prompts=np.array(SWEEP_N))


SWEEP = cached('sweep', _sweep)
GRID = SWEEP['grid']
NAMED = SWEEP['named'].astype(np.float64)                 # (y1*, y2*, y3*) fraction scored
VALID = NAMED > 0                                         # False on the skipped repeated-value cells
PFIELD = np.where(VALID[None], SWEEP['P'].astype(np.float64), np.nan)   # (slot, y1*, y2*, y3*)
# Renormalise to "given it named one of its three numbers", so the three fields sum to 1 and a
# cell where the model went off-prompt does not read as a weak preference.
PCOND = PFIELD / np.where(VALID, NAMED, np.nan)[None]
print(f"grid {GRID.tolist()}   case {SWEEP['case']}   {int(SWEEP['n_prompts'])} prompts/cell")
print(f'the model named one of its own three numbers in {NAMED[VALID].mean():.3f} of '
      f'scored prompts ({VALID.sum()} of {NAMED.size} cells scored)')

### Figure — slices through $y_3^\ast$, and the slot-3 takeover surface

In [ ]:
SLICE_K = [0, 2, 4, 6, 8]           # indices into GRID for the y3* slices
SLOT_COL3 = [NCOL[1], NCOL[2], NCOL[3]]
cmap3 = matplotlib.colors.ListedColormap(SLOT_COL3)

fig = plt.figure(figsize=(4.0 * len(SLICE_K) + 5.5, 4.8))
gs = fig.add_gridspec(1, len(SLICE_K) + 1, width_ratios=[1] * len(SLICE_K) + [1.4],
                      wspace=0.3, right=0.925)
_h = 0.5 * (GRID[1] - GRID[0])                 # imshow extent is cell edges, GRID holds centres
ext = [GRID[0] - _h, GRID[-1] + _h, GRID[0] - _h, GRID[-1] + _h]
for c, k in enumerate(SLICE_K):
    ax = fig.add_subplot(gs[0, c])
    sl = PCOND[:, :, :, k]                               # (slot, y1*, y2*)
    ok = np.isfinite(sl[0])
    win = np.where(ok, np.nanargmax(np.where(ok[None], sl, -1), axis=0).astype(float), np.nan)
    ax.imshow(np.ma.masked_invalid(win).T, origin='lower', extent=ext, cmap=cmap3,
              vmin=-0.5, vmax=2.5, alpha=0.30, aspect='auto', interpolation='nearest')
    for j in range(3):
        f = np.where(ok, sl[j], np.nan)
        if np.nanmax(f) > 0.5 > np.nanmin(f):
            ax.contour(GRID, GRID, f.T, levels=[0.5], colors=[SLOT_COL3[j]], linewidths=2.2)
    ax.plot(GRID, GRID, ls=':', color='0.35', lw=1.2)
    ax.set_title(f'$y_3^*$ = {GRID[k]}', fontsize=TITLE_FS - 3)
    ax.set_xlabel('$y_1^*$', fontsize=LABEL_FS)
    if c == 0: ax.set_ylabel('$y_2^*$', fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS)

# Where slot 3 takes over: the smallest y3* at which P(slot 3) crosses 0.5, as a height surface.
ax = fig.add_subplot(gs[0, len(SLICE_K)], projection='3d')
P3 = PCOND[2]
Z = np.full(P3.shape[:2], np.nan)
for i in range(P3.shape[0]):
    for j in range(P3.shape[1]):
        col = P3[i, j]
        good = np.where(np.isfinite(col) & (col > 0.5))[0]
        if len(good):
            Z[i, j] = GRID[good[0]]
G1, G2 = np.meshgrid(GRID, GRID, indexing='ij')
ax.plot_surface(G1, G2, np.ma.masked_invalid(Z), cmap='viridis', edgecolor='0.4', lw=0.3,
                alpha=0.9, rstride=1, cstride=1)
ax.set_xlabel('$y_1^*$', fontsize=LABEL_FS - 2, labelpad=4)
ax.set_ylabel('$y_2^*$', fontsize=LABEL_FS - 2, labelpad=4)
ax.set_zlabel('$y_3^*$ at takeover', fontsize=LABEL_FS - 3, labelpad=8)
ax.set_title('slot 3 wins above this surface', fontsize=TITLE_FS - 3)
ax.tick_params(labelsize=9, pad=1); ax.view_init(elev=24, azim=-60)

fig.suptitle('implanted coordinates decide which slot is named  (fill: argmax slot; '
             'lines: $P=0.5$)', fontsize=TITLE_FS)
savefig(fig, 'space_contours'); plt.show()

hit = n = 0
for i, y1 in enumerate(GRID):
    for j, y2 in enumerate(GRID):
        for k, y3 in enumerate(GRID):
            if not VALID[i, j, k]:
                continue
            hit += int(np.argmax(PCOND[:, i, j, k]) == int(np.argmax([y1, y2, y3])))
            n += 1
print(f'the named slot matches the largest implanted number in {hit}/{n} ({hit / n:.3f}) '
      f'of scored cells  (chance 1/3)')

# 6. Freezing MLP14 and MLP15

Base patch is the rank-3 span at the pre-MLP L14 residual. A freeze pins a sub-block's output at
$n_3$ to its unpatched corrupted value, so the patch cannot reach the rest of the network through
it.

In [ ]:
SWEEP_LAYERS = [14, 15, 16]
FREEZE_SETS = {'none': ()}
FREEZE_SETS.update({f'MLP{l}': (('mlp', l),) for l in SWEEP_LAYERS})
FREEZE_SETS.update({f'attn{l}': (('attn', l),) for l in SWEEP_LAYERS})
FREEZE_SETS['MLP14+15'] = (('mlp', 14), ('mlp', 15))
SITES = [(k, l) for l in SWEEP_LAYERS for k in ('mlp', 'attn')]

FRZ = {}


def get_frz(o):
    if o not in FRZ:
        FRZ[o] = C.capture_sites(SITES, get_bat(o)['ids_r'], POS3)
    return FRZ[o]


def freeze_extra(o, fz):
    src = get_frz(o)
    return lambda lo, hi: [C.block_of(s).register_forward_hook(
        C.make_out_patch_hook(src[s][lo:hi], POS3)) for s in fz]


def _freeze():
    out = {}
    Pt = torch.tensor(PB, device=device)
    for case in CASE_LIST:
        o = CASES[case]
        bat, src = get_bat(o), get_src(o, POS3)
        for nm, fz in FREEZE_SETS.items():
            rec, iia = MC.eval_spec(dict(premlp=Pt), bat, src, POS3, extra=freeze_extra(o, fz))
            out[f'posrec_{case}_{nm}'] = rec.astype(np.float32)
            out[f'iia_{case}_{nm}'] = iia.astype(np.float32)
        print(f'{case}: ' + '  '.join(f'{nm} {out[f"iia_{case}_{nm}"].mean():.2f}'
                                      for nm in FREEZE_SETS), flush=True)
    out['freeze_names'] = np.array(list(FREEZE_SETS))
    return out


FREEZE = cached('freeze', _freeze)

In [ ]:
fz_names = list(FREEZE['freeze_names'])
fig, axes = plt.subplots(1, 2, figsize=(17, 5.6))
for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.15)),
                                        ('posrec', 'position recovery', None)]):
    xs, W = np.arange(len(fz_names)), 0.8 / len(CASE_LIST)
    for ci, case in enumerate(CASE_LIST):
        v = np.array([FREEZE[f'{key}_{case}_{nm}'].mean() for nm in fz_names])
        e = np.array([FREEZE[f'{key}_{case}_{nm}'].std()
                      / np.sqrt(len(FREEZE[f'{key}_{case}_{nm}'])) for nm in fz_names])
        ax.bar(xs + (ci - (len(CASE_LIST) - 1) / 2) * W, v, W, yerr=e, capsize=3,
               color=CASE_COL[case], alpha=0.92, label=f'{Y[int(case[1])]} perturbed')
        ax.axhline(FREEZE[f'{key}_{case}_none'].mean(), ls='--', lw=1.1,
                   color=CASE_COL[case], alpha=0.7)
    ax.set_xticks(xs); ax.set_xticklabels(fz_names, rotation=30, ha='right', fontsize=TICK_FS)
    ax.axhline(0, color='0.6', lw=0.9)
    if ylim: ax.set_ylim(*ylim)
    style_axes(ax, ylabel=ylab, xlabel='frozen sub-block')
    ax.legend(fontsize=LEGEND_FS)
fig.suptitle('freezing under the rank-3 span patch (dashed: that case unfrozen)',
             fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'module_freezes'); plt.show()

# 7. Which neurons — attribution, then top-$k$ freezing

$s_i \approx (a_i^{\text{corr}} - a_i^{\text{patched}})\,\partial\,\text{PLD}/\partial a_i$,
evaluated on the patched run, over MLP14 + MLP15. Ranked on the 128 training quadruples, verified
by freezing on the 400 eval ones.

In [ ]:
K_GRID = [1, 3, 10, 30, 100, 300, 1000, 3000]
RAND_SEEDS = [0, 1]
N_PROFILE = 20          # per-case depth kept from each ranking
N_CONSENSUS = 12        # neurons in the "matters in all three cases" set (matches the 12 the
                        # two-number analysis identified, so the two sets are comparable)
N_POOLED = 2 * D_FF

NEUR = {}


def get_neur(o):
    if o not in NEUR:
        NEUR[o] = C.capture_neurons(get_bat(o)['ids_r'], NEURON_LAYERS, POS3)
    return NEUR[o]


def neuron_sel(pooled):
    pooled = np.asarray(pooled)
    return {l: torch.as_tensor(POOL_WITHIN[pooled[POOL_LAYER[pooled] == l]],
                               device=device, dtype=torch.long) for l in NEURON_LAYERS}


def freeze_neurons(case, pooled, base=None):
    '''IIA / position recovery with `pooled` neurons pinned to their corrupted values.'''
    o = CASES[case]
    sel, corr = neuron_sel(pooled), get_neur(o)
    Pt = torch.tensor(PB, device=device)

    def extra(lo, hi):
        return [C.mlp_of(l).down_proj.register_forward_pre_hook(
            C.make_neuron_freeze_hook(ii, corr[l][lo:hi][:, ii], POS3))
            for l, ii in sel.items() if len(ii)]
    return MC.eval_spec(base if base is not None else dict(premlp=Pt),
                        get_bat(o), get_src(o, POS3), POS3, extra=extra)


def _attr():
    out = {}
    Pt = torch.tensor(PB, device=device)
    for case in CASE_LIST:
        o = CASES[case]
        tbat = MC.make_batch(TRAIN_Q, o, T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        tsrc = MC.sources_for(tbat, POS3)
        a = MC.attribution(dict(premlp=Pt), tbat, tsrc, POS3, NEURON_LAYERS)
        for l in NEURON_LAYERS:
            out[f'attr_{case}_L{l}'] = a[l]
        del tsrc, tbat
        torch.cuda.empty_cache()
        print(f'{case}: most negative score {min(a[l].min() for l in NEURON_LAYERS):+.4f}',
              flush=True)

    SC = {c: np.concatenate([out[f'attr_{c}_L{l}'] for l in NEURON_LAYERS]) for c in CASE_LIST}
    ORD = {c: np.argsort(SC[c]) for c in CASE_LIST}
    for c in CASE_LIST:
        out[f'order_{c}'] = ORD[c][:1000].astype(np.int64)

    for case in CASE_LIST:
        for k in K_GRID:
            series = {'top': ORD[case][:k], 'bottom': ORD[case][-k:]}
            for s in RAND_SEEDS:
                series[f'rand{s}'] = np.random.default_rng(s).choice(N_POOLED, k, replace=False)
            for nm, pooled in series.items():
                rec, iia = freeze_neurons(case, pooled)
                out[f'topk_posrec_{case}_{nm}_k{k}'] = rec.astype(np.float32)
                out[f'topk_iia_{case}_{nm}_k{k}'] = iia.astype(np.float32)
        print(f'{case}: k=10 IIA {out[f"topk_iia_{case}_top_k10"].mean():.3f}', flush=True)

    pos_in = {}
    for c in CASE_LIST:
        p = np.empty(N_POOLED, dtype=np.int64); p[ORD[c]] = np.arange(N_POOLED); pos_in[c] = p

    # Strict intersections, recorded for the record. A three-way top-20 intersection is expected
    # to be empty by chance (20^3 / 37888^2 ~ 6e-6), so it is reported rather than used.
    for a_, b_ in [('y1', 'y2'), ('y1', 'y3'), ('y2', 'y3')]:
        out[f'shared_{a_}{b_}'] = np.array(
            sorted(set(ORD[a_][:N_PROFILE]) & set(ORD[b_][:N_PROFILE])), dtype=np.int64)
    out['shared_all'] = np.array(sorted(set(ORD['y1'][:N_PROFILE]) & set(ORD['y2'][:N_PROFILE])
                                        & set(ORD['y3'][:N_PROFILE])), dtype=np.int64)

    # The set the freeze test uses: best *worst-case* rank across the three perturbations, i.e.
    # neurons that matter whichever number moved. Always non-empty, and it is the direct analogue
    # of the two-number overlap set (which was an intersection only because there were two cases).
    worst = np.max(np.stack([pos_in[c] for c in CASE_LIST]), axis=0)
    consensus = np.argsort(worst)[:N_CONSENSUS].astype(np.int64)
    out['consensus'] = consensus
    out['consensus_worst_rank'] = (worst[consensus] + 1).astype(np.int64)

    union = sorted(set(consensus.tolist()).union(*[set(ORD[c][:N_PROFILE]) for c in CASE_LIST]),
                   key=lambda i: min(pos_in[c][i] for c in CASE_LIST))
    out['prof_pooled'] = np.array(union, dtype=np.int64)
    for c in CASE_LIST:
        out[f'prof_rank_{c}'] = (pos_in[c][np.array(union)] + 1).astype(np.int64)
    print(f"strict 3-way overlap: {len(out['shared_all'])}   consensus set: {len(consensus)}   "
          f'profiled union: {len(union)}', flush=True)

    rng = np.random.default_rng(100)
    for case in CASE_LIST:
        for nm, pooled in (('consensus', consensus),
                           ('rand0', rng.choice(N_POOLED, len(consensus), replace=False)),
                           ('rand1', rng.choice(N_POOLED, len(consensus), replace=False))):
            rec, iia = freeze_neurons(case, pooled)
            out[f'sh_posrec_{case}_{nm}'] = rec.astype(np.float32)
            out[f'sh_iia_{case}_{nm}'] = iia.astype(np.float32)
    out['k_grid'] = np.array(K_GRID)
    return out


ATTR = cached('attr', _attr)
PROF = ATTR['prof_pooled']
CONSENSUS = ATTR['consensus']
print(f'profiled {len(PROF)} neurons  (L14 {(POOL_LAYER[PROF] == 14).sum()}, '
      f'L15 {(POOL_LAYER[PROF] == 15).sum()})')
print(f'strict 3-way top-{N_PROFILE} overlap: '
      + (', '.join(NAME(p) for p in ATTR['shared_all']) if len(ATTR['shared_all']) else 'empty'))
print(f'consensus set ({len(CONSENSUS)}, by best worst-case rank): '
      + ', '.join(f'{NAME(p)}({r:,})' for p, r in zip(CONSENSUS, ATTR['consensus_worst_rank'])))

In [ ]:
SER = [('top', 'top-$k$ by attribution', COL['full'], '-'),
       ('bottom', 'bottom-$k$', COL['pc1'], '-'),
       ('rand0', 'random $k$', COL['ceil'], '--'),
       ('rand1', '_nolegend_', COL['ceil'], '--')]

fig, axes = plt.subplots(2, len(CASE_LIST), figsize=(6.0 * len(CASE_LIST), 9.2), squeeze=False)
for ci, case in enumerate(CASE_LIST):
    for ri, (key, ylab) in enumerate([('iia', 'IIA'), ('posrec', 'position recovery')]):
        ax = axes[ri, ci]
        for nm, lab, col, ls in SER:
            v = np.array([ATTR[f'topk_{key}_{case}_{nm}_k{k}'].mean() for k in K_GRID])
            ax.plot(K_GRID, v, ls, marker='o', ms=4.5, color=col, lw=1.8, label=lab)
        ax.axhline(FREEZE[f'{key}_{case}_none'].mean(), ls=':', color='0.35', lw=1.6,
                   label='no freeze')
        ax.axhline(FREEZE[f'{key}_{case}_MLP14+15'].mean(), ls='-.', color='0.35', lw=1.4,
                   label='MLP14+15 frozen')
        ax.set_xscale('log')
        style_axes(ax, xlabel='$k$ neurons frozen' if ri else None, ylabel=ylab if ci == 0 else None,
                   title=f'{Y[int(case[1])]} perturbed' if ri == 0 else None)
        ax.grid(alpha=0.3)
        if ri == 0 and ci == 0:
            ax.legend(fontsize=LEGEND_FS - 1)
fig.suptitle('freezing the top-ranked neurons destroys the patch', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'topk_freeze'); plt.show()

fig, ax = plt.subplots(figsize=(8.2, 5.0))
xs, W = np.arange(len(CASE_LIST)), 0.24
for bi, (nm, lab, col) in enumerate(
        [('none', 'no freeze', COL['ceil']),
         ('consensus', f'{len(CONSENSUS)} consensus neurons frozen', COL['full']),
         ('rand0', f'{len(CONSENSUS)} random frozen', COL['pc1']),
         ('rand1', '_nolegend_', COL['pc1'])]):
    vals = [(FREEZE[f'iia_{c}_none'] if nm == 'none' else ATTR[f'sh_iia_{c}_{nm}'])
            for c in CASE_LIST]
    ax.bar(xs + (bi - 1.5) * W, [v.mean() for v in vals], W,
           yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col,
           alpha=0.6 if nm == 'rand1' else 0.92, label=lab)
ax.set_xticks(xs)
ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS)
style_axes(ax, ylabel='IIA')
ax.legend(fontsize=LEGEND_FS)
ax.set_title('freezing the neurons that matter in all three cases', fontsize=TITLE_FS)
plt.tight_layout(); savefig(fig, 'consensus_freeze'); plt.show()

# 8. Receptive fields

Post-SwiGLU activation over a dense $(y_1, y_2, y_3)$ cube at $n_3$, shown as the three pairwise
marginals (each averaged over the number not on the axes).

In [ ]:
RF_STEP = 4
RF_GRID = np.arange(LOW + 1, HIGH, RF_STEP)

TRANSFER = TWO['overlap_k20'] if TWO is not None else np.array([], dtype=np.int64)
RF_POOLED = np.array(sorted(set(PROF.tolist()) | set(TRANSFER.tolist())), dtype=np.int64)


def _rf():
    idx = {l: torch.as_tensor(POOL_WITHIN[RF_POOLED[POOL_LAYER[RF_POOLED] == l]],
                              device=device, dtype=torch.long) for l in NEURON_LAYERS}
    trips, acts = MC.receptive_fields(RF_GRID, T, POS3, NEURON_LAYERS, idx)
    out = {'rf_grid': RF_GRID.astype(np.int64), 'rf_pooled': RF_POOLED, 'rf_trips': trips}
    for l in NEURON_LAYERS:
        out[f'rf_acts_L{l}'] = acts[l]
        out[f'rf_idx_L{l}'] = idx[l].cpu().numpy()
    return out


RF = cached('rf', _rf)
G = len(RF['rf_grid'])
RF_COL = {p: (int(POOL_LAYER[p]), int(np.where(RF[f'rf_idx_L{POOL_LAYER[p]}']
                                               == POOL_WITHIN[p])[0][0])) for p in RF['rf_pooled']}


def rf_cube(pooled):
    '''(G, G, G) activation cube for one pooled neuron, axes (y1, y2, y3).'''
    l, c = RF_COL[pooled]
    return RF[f'rf_acts_L{l}'][:, c].reshape(G, G, G)


PAIRS = [(2, Y[1], Y[2]), (1, Y[1], Y[3]), (0, Y[2], Y[3])]     # (axis averaged out, x, y)
SPACER = 0.32                # width of the gap column that separates one neuron's block


def draw_marginals(pooled_list, fname, title, ranks=None, ncol_neurons=4):
    '''Three pairwise marginals per neuron, in blocks of three columns separated by a gap.'''
    n = len(pooled_list)
    ncol = min(ncol_neurons, n)
    nrow = int(np.ceil(n / ncol))
    g = RF['rf_grid']
    h = 0.5 * (g[1] - g[0])                     # imshow extent is cell edges, the grid is centres
    ext = [g[0] - h, g[-1] + h, g[0] - h, g[-1] + h]

    widths = []
    for b in range(ncol):
        widths += [1, 1, 1] + ([SPACER] if b < ncol - 1 else [])
    fig = plt.figure(figsize=(2.3 * sum(widths), 2.95 * nrow))
    gs = fig.add_gridspec(nrow, len(widths), width_ratios=widths, wspace=0.42, hspace=0.62)
    for slot in range(n):
        r, b = slot // ncol, slot % ncol
        c0 = b * 4
        p = pooled_list[slot]
        cube = rf_cube(p)
        # One symmetric scale per neuron, taken on the marginals (not the cube): averaging out a
        # number shrinks the range, and a per-panel scale would hide which pair it reads.
        mars = [cube.mean(axis=a) for a, _, _ in PAIRS]
        vmax = max(np.abs(m).max() for m in mars) + 1e-9
        axs = []
        for k, (_avg, xl, yl) in enumerate(PAIRS):
            ax = fig.add_subplot(gs[r, c0 + k]); axs.append(ax)
            ax.imshow(mars[k].T, origin='lower', extent=ext, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                      aspect='auto', interpolation='nearest')
            ax.plot(g, g, ls=':', color='0.3', lw=0.9)
            ax.set_xlabel(xl, fontsize=12, labelpad=1)
            ax.set_ylabel(yl, fontsize=12, labelpad=0)
            ax.tick_params(labelsize=8)
        # One heading per neuron, centred over its own three panels.
        x0 = axs[0].get_position().x0
        rk = '' if ranks is None else f" (rank {ranks.get(int(p), '-')})"
        fig.text(0.5 * (x0 + axs[-1].get_position().x1),
                 axs[0].get_position().y1 + 0.012, f'{NAME(p)}{rk}',
                 ha='center', va='bottom', fontsize=13, fontweight='bold')
    fig.suptitle(title + '   (diverging scale, one per neuron)', fontsize=TITLE_FS, y=1.0)
    savefig(fig, fname); plt.show()

In [ ]:
N_SHOW = 8
RANKS = {int(p): int(min(ATTR[f'prof_rank_{c}'][i] for c in CASE_LIST))
         for i, p in enumerate(PROF)}
draw_marginals(list(PROF[:N_SHOW]), 'rf_profiled',
               'top-ranked neurons: pairwise marginals of the $(y_1, y_2, y_3)$ field',
               ranks=RANKS)

# 9. MLP14 $\to$ MLP15 connections

**Causal edge** — one MLP14 neuron set to its clean value at $n_3$, read as the shift in each
MLP15 neuron in units of that neuron's cloud SD. **Weight path** — the write column against the
$\gamma$-folded read row.

In [ ]:
def _connect():
    out = {}
    i14 = POOL_WITHIN[PROF[POOL_LAYER[PROF] == 14]]
    i15 = POOL_WITHIN[PROF[POOL_LAYER[PROF] == 15]]
    t14 = torch.as_tensor(i14, device=device, dtype=torch.long)
    t15 = torch.as_tensor(i15, device=device, dtype=torch.long)
    out['i14'], out['i15'] = i14.astype(np.int64), i15.astype(np.int64)

    Wd14 = C.mlp_of(14).down_proj.weight.detach().float()
    g15 = D.model.model.layers[15].post_attention_layernorm.weight.detach().float()
    A = Wd14[:, t14]
    An = A / A.norm(dim=0, keepdim=True)
    for which, Wr in (('gate', C.mlp_of(15).gate_proj.weight.detach().float() * g15),
                      ('up', C.mlp_of(15).up_proj.weight.detach().float() * g15)):
        Rn = Wr / Wr.norm(dim=1, keepdim=True)
        allc = (Rn @ An).cpu().numpy()
        out[f'path_{which}'] = allc[i15].T.astype(np.float32)     # (n14, n15)
        out[f'path_{which}_null'] = allc.astype(np.float32)

    # Cloud SD of each L15 target, so an edge is in units the neuron actually varies over.
    cl15 = C.capture_neurons(
        D.tokenizer([MC.prompt_gen(list(t)) for t in MC.sample_cloud_triples(400, SEED + 3,
                                                                             LOW, HIGH)],
                    return_tensors='pt')['input_ids'].to(device),
        [15], POS3, idx={15: t15})[15].cpu().numpy()
    sd15 = cl15.std(0) + 1e-6
    out['sd15'] = sd15.astype(np.float32)

    for case in CASE_LIST:
        o = CASES[case]
        bat = get_bat(o)
        a_cl = C.capture_neurons(bat['ids_c'], [14], POS3, idx={14: t14})[14]
        base = C.capture_neurons(bat['ids_r'], [15], POS3, idx={15: t15})[15].cpu().numpy()
        rows = []
        for c in range(len(i14)):
            got = C.capture_neurons(
                bat['ids_r'], [15], POS3, idx={15: t15},
                patch=lambda lo, hi, _c=c, _a=a_cl: [
                    C.mlp_of(14).down_proj.register_forward_pre_hook(
                        C.make_neuron_freeze_hook(t14[_c:_c + 1], _a[lo:hi, _c:_c + 1],
                                                  POS3))])[15].cpu().numpy()
            rows.append((got - base).mean(0) / sd15)
        out[f'edge_{case}'] = np.array(rows, dtype=np.float32)
        print(f'{case}: strongest edge |{np.abs(out[f"edge_{case}"]).max():.3f}| SD', flush=True)
        del a_cl
        torch.cuda.empty_cache()
    return out


CONN = cached('connect', _connect)
I14, I15 = CONN['i14'], CONN['i15']
P14 = np.array([np.where((POOL_LAYER == 14) & (POOL_WITHIN == w))[0][0] for w in I14])
P15 = np.array([np.where((POOL_LAYER == 15) & (POOL_WITHIN == w))[0][0] for w in I15])

In [ ]:
EDGE = np.mean([CONN[f'edge_{c}'] for c in CASE_LIST], axis=0)      # (n14, n15)
lim = np.abs(EDGE).max()

fig, axes = plt.subplots(1, 2, figsize=(16.5, 1.0 + 0.42 * len(I14)),
                         width_ratios=[1, 1])
for ax, (Mx, ttl, cm, vl) in zip(axes, [
        (EDGE, 'causal edge (SD of the L15 neuron), mean over cases', 'RdBu_r', lim),
        (CONN['path_gate'], r'weight path: $w^{\rm down}_{14}\cdot\hat w^{\rm gate}_{15}$',
         'PuOr_r', np.abs(CONN['path_gate']).max())]):
    im = ax.imshow(Mx, cmap=cm, vmin=-vl, vmax=vl, aspect='auto', interpolation='nearest')
    ax.set_xticks(range(len(I15)))
    ax.set_xticklabels([f'#{w}' for w in I15], rotation=90, fontsize=9)
    ax.set_yticks(range(len(I14)))
    ax.set_yticklabels([f'#{w}' for w in I14], fontsize=9)
    ax.set_title(ttl, fontsize=TITLE_FS - 3)
    ax.set_xlabel('MLP15 neuron', fontsize=LABEL_FS)
    ax.set_ylabel('MLP14 neuron', fontsize=LABEL_FS)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
plt.tight_layout(); savefig(fig, 'connectivity'); plt.show()

TGT = int(np.argmax(np.abs(EDGE).sum(0)))
SRC_ORD = np.argsort(-np.abs(EDGE[:, TGT]))[:4]
print(f'strongest-driven MLP15 target: {NAME(P15[TGT])}')
for r in SRC_ORD:
    print(f'  fed by {NAME(P14[r]):>12}  edge {EDGE[r, TGT]:+.3f} SD   '
          f'path(gate) {CONN["path_gate"][r, TGT]:+.4f}')

### Receptive fields of the sources feeding that target

In [ ]:
draw_marginals([int(P14[r]) for r in SRC_ORD] + [int(P15[TGT])], 'rf_connected',
               f'MLP14 sources (left) feeding {NAME(P15[TGT])} (right)', ncol_neurons=5)

# 10. The two-number neurons in the three-number task

The 12 neurons `makefig_comparator` identified as shared between its two cases
(`overlap_k20` in `results/comparator_data_2digit.npz`), re-measured here: their fields over three
numbers, and what freezing them does to the rank-3 span patch.

In [ ]:
def _transfer():
    out = {}
    Pt = torch.tensor(PB, device=device)
    if not len(TRANSFER):
        return {'none': np.array(0)}
    rng = np.random.default_rng(7)
    sets = {'two_number': TRANSFER,
            'rand0': rng.choice(N_POOLED, len(TRANSFER), replace=False),
            'rand1': rng.choice(N_POOLED, len(TRANSFER), replace=False)}
    for case in CASE_LIST:
        for nm, pooled in sets.items():
            rec, iia = freeze_neurons(case, pooled)
            out[f'posrec_{case}_{nm}'] = rec.astype(np.float32)
            out[f'iia_{case}_{nm}'] = iia.astype(np.float32)
        print(f'{case}: two-number set {out[f"iia_{case}_two_number"].mean():.3f}   '
              f'random {out[f"iia_{case}_rand0"].mean():.3f}   '
              f'unfrozen {FREEZE[f"iia_{case}_none"].mean():.3f}', flush=True)
    # Where they rank under the three-number attribution.
    out['pooled'] = TRANSFER
    for c in CASE_LIST:
        full = np.concatenate([ATTR[f'attr_{c}_L{l}'] for l in NEURON_LAYERS])
        rank = np.empty(N_POOLED, dtype=np.int64)
        rank[np.argsort(full)] = np.arange(N_POOLED)
        out[f'rank3_{c}'] = (rank[TRANSFER] + 1).astype(np.int64)
    return out


TRANS = cached('transfer', _transfer)
if len(TRANSFER):
    print(f'{"neuron":>12}' + ''.join(f'{c + " rank":>12}' for c in CASE_LIST))
    for i, p in enumerate(TRANSFER):
        print(f'{NAME(p):>12}' + ''.join(f'{int(TRANS[f"rank3_{c}"][i]):>12,}'
                                         for c in CASE_LIST))

In [ ]:
if len(TRANSFER):
    draw_marginals(list(TRANSFER), 'rf_two_number_neurons',
                   'the two-number comparator neurons, measured on three numbers',
                   ncol_neurons=4)

In [ ]:
if len(TRANSFER):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
    for ax, (key, ylab, ylim) in zip(axes, [('iia', 'IIA', (0, 1.12)),
                                            ('posrec', 'position recovery', None)]):
        xs, W = np.arange(len(CASE_LIST)), 0.26
        bars = [('none', 'no freeze', COL['ceil']),
                ('two_number', f'the {len(TRANSFER)} two-number neurons', COL['full']),
                ('rand0', f'{len(TRANSFER)} random neurons', COL['pc1'])]
        for bi, (nm, lab, col) in enumerate(bars):
            vals = [(FREEZE[f'{key}_{c}_none'] if nm == 'none' else TRANS[f'{key}_{c}_{nm}'])
                    for c in CASE_LIST]
            ax.bar(xs + (bi - 1) * W, [v.mean() for v in vals], W,
                   yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col,
                   alpha=0.92, label=lab)
        ax.set_xticks(xs)
        ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS)
        ax.axhline(0, color='0.6', lw=0.9)
        if ylim: ax.set_ylim(*ylim)
        style_axes(ax, ylabel=ylab)
        ax.legend(fontsize=LEGEND_FS)
    fig.suptitle('freezing the two-number neurons under the three-number span patch',
                 fontsize=TITLE_FS)
    plt.tight_layout(); savefig(fig, 'transfer_freeze'); plt.show()

# 11. L15 — the comparison outcome

DAS at rank 1 and 2 on the residual leaving L15 at $n_3$, and the PCA of the same cloud, every
projection coloured by **which slot holds the maximum**. Section 3's space carries the three
numbers; this asks what the layer above carries.

Cost: `len(DAS15_CASES) x len(DAS15_RANKS)` DAS fits, each the price of one section-2 fit. Trim
`DAS15_CASES` to a single case to cut it.

In [ ]:
DAS15_LAYER = 15
DAS15_RANKS = [1, 2]
DAS15_CASES = CASE_LIST        # cases the subspace is fitted on
DAS15_SEED  = 0


def src_with_L15(o):
    '''Section 3's sources plus the L15 residual, added in place so nothing is re-captured.'''
    s = get_src(o, POS3)
    if DAS15_LAYER not in s:
        s[DAS15_LAYER] = S.clean_hidden_at(get_bat(o)['ids_c'], DAS15_LAYER)
    return s


def _das15():
    out = {}
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    assert np.array_equal(np.array(trips, dtype=np.int64), YV), \
        'the L15 cloud must be the same triples as the section-2 cloud'
    c15 = MC.capture_resid_cloud(trips, T, [(DAS15_LAYER, POS3)])[(DAS15_LAYER, POS3)]
    out['cloud15'] = c15

    # pc1_of fits a 3-component PCA and hands it back, so no extra sklearn import is needed here.
    _, pca15 = MC.pc1_of(c15)
    out['pca15_coords'] = pca15.transform(c15).astype(np.float32)
    out['pca15_components'] = pca15.components_.astype(np.float32)
    out['pca15_evr'] = pca15.explained_variance_ratio_.astype(np.float32)
    print(f'  L15 PCA EVR {np.round(pca15.explained_variance_ratio_, 3).tolist()}', flush=True)

    bases = {}
    for case in DAS15_CASES:
        tbat = MC.make_batch(TRAIN_Q, CASES[case], T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        ch = S.clean_hidden_at(tbat['ids_c'], DAS15_LAYER)
        for k in DAS15_RANKS:
            Q, curve = MC.train_resid_das(DAS15_LAYER, tbat, ch, POS3, k, seed=DAS15_SEED)
            bases[(case, k)] = Q
            out[f'basis_{case}_k{k}'] = Q.cpu().numpy().astype(np.float32)
            out[f'curve_{case}_k{k}'] = curve
            out[f'proj_{case}_k{k}'] = (c15 @ Q.cpu().numpy()).astype(np.float32)
            print(f'  L15 DAS {case} k={k}: loss {curve[0]:.3f} -> {curve[-1]:.3f}', flush=True)
            # Cross-evaluation: a subspace fitted on one perturbation, scored on all three. If L15
            # holds one shared "which slot won" variable, the off-diagonal should hold up.
            for tc in CASE_LIST:
                rec, iia = MC.eval_spec(dict(resid=[(DAS15_LAYER, 'sub', Q)]),
                                        get_bat(CASES[tc]), src_with_L15(CASES[tc]), POS3)
                out[f'posrec_{tc}_from_{case}_k{k}'] = rec.astype(np.float32)
                out[f'iia_{tc}_from_{case}_k{k}'] = iia.astype(np.float32)
        del ch, tbat
        torch.cuda.empty_cache()

    for tc in CASE_LIST:                       # full-rank ceiling at the same site
        rec, iia = MC.eval_spec(dict(resid=[(DAS15_LAYER, 'full', None)]),
                                get_bat(CASES[tc]), src_with_L15(CASES[tc]), POS3)
        out[f'posrec_{tc}_full'] = rec.astype(np.float32)
        out[f'iia_{tc}_full'] = iia.astype(np.float32)

    # rho_j = ||Q^T w_j|| / ||w_j|| for EVERY MLP15 neuron, so the profiled ones can be placed in
    # their own empirical null (down_proj columns are not isotropic, so sqrt(r/d) is not a
    # trustworthy chance level).
    out.update(C.das15_write_alignment(bases, layer=DAS15_LAYER))
    return out


DAS15 = cached('das15', _das15)
ARGMAX_SLOT = YV.argmax(1)
print('cloud argmax slot counts:', np.bincount(ARGMAX_SLOT, minlength=3).tolist())

### Figure — the L15 cloud, coloured by which slot holds the maximum

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

DAS15_SHOW = 'y1'                 # which fitted subspace to draw
SLOTC = [NCOL[1], NCOL[2], NCOL[3]]
SLOT_LAB = {0: 'max at $s_1$', 1: 'max at $s_2$', 2: 'max at $s_3$'}

p1 = DAS15[f'proj_{DAS15_SHOW}_k1'][:, 0]
p2 = DAS15[f'proj_{DAS15_SHOW}_k2']
pc, evr = DAS15['pca15_coords'], DAS15['pca15_evr']

fig = plt.figure(figsize=(22, 5.6))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.12], wspace=0.30)

ax = fig.add_subplot(gs[0, 0])
bins = np.linspace(p1.min(), p1.max(), 46)
for s in range(3):
    ax.hist(p1[ARGMAX_SLOT == s], bins=bins, color=SLOTC[s], alpha=0.55, label=SLOT_LAB[s])
style_axes(ax, xlabel='rank-1 DAS component', ylabel='prompts', grid=False)
ax.grid(alpha=0.3); ax.legend(fontsize=LEGEND_FS)
ax.set_title(f'L15 DAS $k=1$  (fitted on {DAS15_SHOW})', fontsize=TITLE_FS - 2)

ax = fig.add_subplot(gs[0, 1])
for s in range(3):
    m = ARGMAX_SLOT == s
    ax.scatter(p2[m, 0], p2[m, 1], s=9, color=SLOTC[s], alpha=0.6, edgecolors='none',
               label=SLOT_LAB[s])
style_axes(ax, xlabel='DAS 1', ylabel='DAS 2', grid=False)
ax.grid(alpha=0.3); ax.legend(fontsize=LEGEND_FS, markerscale=2)
ax.set_title(f'L15 DAS $k=2$  (fitted on {DAS15_SHOW})', fontsize=TITLE_FS - 2)

ax = fig.add_subplot(gs[0, 2])
for s in range(3):
    m = ARGMAX_SLOT == s
    ax.scatter(pc[m, 0], pc[m, 1], s=9, color=SLOTC[s], alpha=0.6, edgecolors='none')
style_axes(ax, xlabel=f'PC1 ({100 * evr[0]:.0f}%)', ylabel=f'PC2 ({100 * evr[1]:.0f}%)', grid=False)
ax.grid(alpha=0.3)
ax.set_title('L15 PCA', fontsize=TITLE_FS - 2)

ax = fig.add_subplot(gs[0, 3], projection='3d')
for s in range(3):
    m = ARGMAX_SLOT == s
    ax.scatter(pc[m, 0], pc[m, 1], pc[m, 2], s=7, color=SLOTC[s], alpha=0.55, edgecolors='none')
ax.set_box_aspect((1, 1, 0.92)); ax.view_init(elev=22, azim=-58)
ax.set_xlabel(f'PC1 ({100 * evr[0]:.0f}%)', fontsize=LABEL_FS - 3, labelpad=5)
ax.set_ylabel(f'PC2 ({100 * evr[1]:.0f}%)', fontsize=LABEL_FS - 3, labelpad=5)
ax.set_zlabel(f'PC3 ({100 * evr[2]:.0f}%)', fontsize=LABEL_FS - 3, labelpad=3)
ax.tick_params(labelsize=8, pad=0.5)
ax.set_title('L15 PCA, 3D', fontsize=TITLE_FS - 2)

fig.suptitle('the L15 residual at $n_3$, coloured by which slot holds the maximum',
             fontsize=TITLE_FS)
savefig(fig, 'das15_spaces'); plt.show()

# How well the slot is linearly readable from each 2-D view -- the number the colours are making.
for nm, X in [('DAS k=2', p2), ('PCA top 2', pc[:, :2]), ('PCA top 3', pc[:, :3])]:
    Xtr, Xte, ytr, yte = train_test_split(X, ARGMAX_SLOT, test_size=0.25, random_state=SEED,
                                          stratify=ARGMAX_SLOT)
    acc = LogisticRegression(max_iter=2000).fit(Xtr, ytr).score(Xte, yte)
    print(f'argmax slot, 3-way held-out accuracy from {nm:>10}: {acc:.3f}   (chance 0.333)')

### Figure — IIA against rank, and across cases

In [ ]:
fig, axes = plt.subplots(1, len(DAS15_RANKS) + 1,
                         figsize=(5.6 * (len(DAS15_RANKS) + 1), 5.0),
                         width_ratios=[1] * len(DAS15_RANKS) + [1.25])
for ax, k in zip(axes, DAS15_RANKS):
    Mx = np.array([[DAS15[f'iia_{tc}_from_{fc}_k{k}'].mean() for tc in CASE_LIST]
                   for fc in DAS15_CASES])
    im = ax.imshow(Mx, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(CASE_LIST)))
    ax.set_xticklabels([f'{Y[int(c[1])]} pert.' for c in CASE_LIST], fontsize=TICK_FS)
    ax.set_yticks(range(len(DAS15_CASES)))
    ax.set_yticklabels([f'fit on {c}' for c in DAS15_CASES], fontsize=TICK_FS)
    for i in range(Mx.shape[0]):
        for j in range(Mx.shape[1]):
            ax.text(j, i, f'{Mx[i, j]:.2f}', ha='center', va='center', fontsize=13,
                    color='white' if Mx[i, j] > 0.55 else 'black')
    ax.set_title(f'L15 DAS rank {k} — IIA', fontsize=TITLE_FS - 2)
    ax.grid(False); ax.tick_params(length=0)
    for s_ in ax.spines.values():
        s_.set_visible(False)

ax = axes[-1]
xs, W = np.arange(len(CASE_LIST)), 0.8 / (len(DAS15_RANKS) + 1)
series = [(f'k={k} (own case)', lambda c, _k=k: DAS15[f'iia_{c}_from_{c}_k{_k}'],
           [COL['h14'], COL['pc1'], COL['das']][i % 3]) for i, k in enumerate(DAS15_RANKS)]
series.append(('L15 full', lambda c: DAS15[f'iia_{c}_full'], COL['ceil']))
for bi, (lab, get, col) in enumerate(series):
    vals = [get(c) for c in CASE_LIST]
    ax.bar(xs + (bi - (len(series) - 1) / 2) * W, [v.mean() for v in vals], W,
           yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col, alpha=0.92,
           label=lab)
ax.set_xticks(xs)
ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS - 2)
ax.set_ylim(0, 1.15)
style_axes(ax, ylabel='IIA')
ax.legend(fontsize=LEGEND_FS)
ax.set_title('rank vs the full-rank ceiling at L15', fontsize=TITLE_FS - 2)
plt.tight_layout(); savefig(fig, 'das15_iia'); plt.show()

for k in DAS15_RANKS:
    for fc in DAS15_CASES:
        print(f'rank {k}, fitted on {fc}: ' + '  '.join(
            f'{tc} {DAS15[f"iia_{tc}_from_{fc}_k{k}"].mean():.3f}' for tc in CASE_LIST))

### Which MLP15 neurons write along the DAS space

$\rho_j = \lVert Q^\top w_j\rVert / \lVert w_j\rVert$, $w_j$ the neuron's `down_proj` column.
The null is every MLP15 neuron; the attribution-profiled ones are marked in it.

In [ ]:
AL_CASE, AL_K = 'y1', 1           # which fitted subspace the alignment is measured against
N_ALIGN_SHOW = 4                  # profiled neurons whose receptive fields are drawn below

frac = DAS15[f'dasfrac_L{DAS15_LAYER}_{AL_CASE}_k{AL_K}'].astype(np.float64)
prof15 = PROF[POOL_LAYER[PROF] == DAS15_LAYER]
w15 = POOL_WITHIN[prof15]
pf = frac[w15]
q95, q99 = np.quantile(frac, [0.95, 0.99])
order15 = np.argsort(-pf)
ALIGNED = prof15[order15[:N_ALIGN_SHOW]]

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.hist(frac, bins=140, color='0.72')
ax.axvline(q95, ls='--', color='0.35', lw=1.3, label='95th pct of all MLP15')
ax.axvline(q99, ls=':', color='0.35', lw=1.3, label='99th pct')
ax.set_yscale('log')
# Label y in axes fraction (x stays in data coords) so the spacing survives the log scale.
tx = ax.get_xaxis_transform()
for r, p in enumerate(prof15[order15]):
    v = frac[POOL_WITHIN[p]]
    col = VCOL[1] if r < N_ALIGN_SHOW else COL['pc1']
    ax.axvline(v, color=col, lw=2.0 if r < N_ALIGN_SHOW else 1.0,
               alpha=0.95 if r < N_ALIGN_SHOW else 0.5)
    if r < N_ALIGN_SHOW:
        ax.text(v, 0.96 - 0.075 * r, f' {NAME(p)}', color=col, fontsize=11,
                fontweight='bold', va='top', transform=tx)
style_axes(ax, xlabel=r'$\|Q^\top w_j\| / \|w_j\|$', ylabel='MLP15 neurons', grid=False)
ax.grid(alpha=0.3)
ax.legend(fontsize=LEGEND_FS)
ax.set_title(f'write alignment with the rank-{AL_K} L15 DAS space (fitted on {AL_CASE})',
             fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, 'das15_alignment'); plt.show()

print(f'null over all {len(frac)} MLP15 neurons: median {np.median(frac):.4f}   '
      f'95th {q95:.4f}   99th {q99:.4f}')
print(f'{"neuron":>12}{"rho":>9}{"pct":>8}' + ''.join(f'{c + " rank":>11}' for c in CASE_LIST))
for p in prof15[order15]:
    v = frac[POOL_WITHIN[p]]
    i = int(np.where(PROF == p)[0][0])
    print(f'{NAME(p):>12}{v:>9.4f}{100 * (frac < v).mean():>7.1f}%'
          + ''.join(f'{int(ATTR[f"prof_rank_{c}"][i]):>11,}' for c in CASE_LIST))

In [ ]:
draw_marginals([int(p) for p in ALIGNED], 'rf_das15_aligned',
               f'MLP15 neurons whose write is most aligned with the rank-{AL_K} L15 DAS space',
               ranks=RANKS, ncol_neurons=4)

# 12. $\mathbf{u}_1, \mathbf{u}_2$ — each number at its own position

$\mathbf{u}_j$ = rank-1 DAS on the L13 residual at $n_j$, fitted on case `yj`: the number *before*
a layer-14 head transports it. $\mathbf{v}_1,\mathbf{v}_2$ are the same two numbers read at $n_3$
*after* transport, and $\mathbf{v}_3$ is already the $y_3$ counterpart of a $\mathbf{u}$ (L13 at
$n_3$), so the five together span both sides of the copy.

In [ ]:
U_SITES = {1: (0, 'y1'), 2: (1, 'y2')}     # u_j: L13 residual at n_j, fitted on case y_j


def _ucos():
    out = {}
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    cl = MC.capture_resid_cloud(trips, T, [(LAYER, POS[pi]) for pi, _ in U_SITES.values()])
    for j, (pi, case) in U_SITES.items():
        tbat = MC.make_batch(TRAIN_Q, CASES[case], T)
        tbat['pld_clean'], tbat['pld_corr'] = MC.pos_anchors(tbat)
        ch = S.clean_hidden_at(tbat['ids_c'], LAYER)
        Q, curve = MC.train_resid_das(LAYER, tbat, ch, POS[pi], 1, seed=0)
        u = Q[:, 0].cpu().numpy()
        r = np.corrcoef(cl[(LAYER, POS[pi])] @ u, np.log(YV[:, j - 1]))[0, 1]
        out[f'u{j}'] = (u * np.sign(r)).astype(np.float32)      # rises with its own number
        out[f'u{j}_corr'] = np.array(abs(r))
        out[f'u{j}_curve'] = curve
        out[f'u{j}_pos'] = np.array(POS[pi])
        out[f'u{j}_comp'] = (cl[(LAYER, POS[pi])] @ u * np.sign(r)).astype(np.float32)
        print(f'  u{j}: L13 @ n{j}, fitted on {case}, loss {curve[0]:.3f} -> {curve[-1]:.3f}, '
              f'|corr| with log y{j} = {abs(r):.3f}', flush=True)
        # Causal at its own site: live in its own case, a no-op in the others (the other cases
        # never change the number sitting at n_j).
        for tc in CASE_LIST:
            rec, iia = MC.eval_spec(dict(resid=[(LAYER, 'sub', Q)]), get_bat(CASES[tc]),
                                    get_src(CASES[tc], POS[pi], layers=(LAYER,)), POS[pi])
            out[f'posrec_{tc}_u{j}'] = rec.astype(np.float32)
            out[f'iia_{tc}_u{j}'] = iia.astype(np.float32)
        print(f'      IIA: ' + '  '.join(f'{tc} {out[f"iia_{tc}_u{j}"].mean():.3f}'
                                         for tc in CASE_LIST), flush=True)
        del ch, tbat
        torch.cuda.empty_cache()
    return out


UC = cached('ucos', _ucos)
UDIR = {j: UC[f'u{j}'].astype(np.float64) for j in U_SITES}

### Figure — cosine similarity across all five directions

In [ ]:
FIVE = [(r'$\mathbf{u}_1$', 'L13 @ $n_1$', UDIR[1], VCOL[1]),
        (r'$\mathbf{u}_2$', 'L13 @ $n_2$', UDIR[2], VCOL[2]),
        (r'$\mathbf{v}_1$', 'H14 out @ $n_3$', VDIR[1], VCOL[1]),
        (r'$\mathbf{v}_2$', 'H18 out @ $n_3$', VDIR[2], VCOL[2]),
        (r'$\mathbf{v}_3$', 'L13 @ $n_3$', VDIR[3], VCOL[3])]
C5 = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
                for _, _, b, _ in FIVE] for _, _, a, _ in FIVE])
PLAIN5 = ['u1', 'u2', 'v1', 'v2', 'v3']

fig, axes = plt.subplots(1, 2, figsize=(18, 6.6), width_ratios=[1.25, 1])

ax = axes[0]
im = ax.imshow(np.where(np.tril(np.ones_like(C5)) > 0, C5, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels([f[0] for f in FIVE], fontsize=18)
ax.set_yticks(range(5))
ax.set_yticklabels([f'{f[0]}\n{f[1]}' for f in FIVE], fontsize=14)
for i in range(5):
    for j in range(i + 1):
        ax.text(j, i, f'{C5[i, j]:+.2f}', ha='center', va='center', fontsize=15,
                color='white' if abs(C5[i, j]) > 0.6 else 'black')
# The two same-number pairs: the number before and after a layer-14 head transports it.
for i, j in [(2, 0), (3, 1)]:
    ax.add_patch(Rectangle((min(i, j) - 0.5, max(i, j) - 0.5), 1, 1, fill=False,
                           edgecolor='#111111', lw=3.0, zorder=6))
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
cb.set_label('cosine similarity', fontsize=LABEL_FS); cb.ax.tick_params(labelsize=TICK_FS)
ax.grid(False); ax.tick_params(length=0)
for s_ in ax.spines.values():
    s_.set_visible(False)
ax.set_title('boxed: the same number, before and after transport', fontsize=TITLE_FS - 1)

ax = axes[1]
xs, W = np.arange(len(CASE_LIST)), 0.8 / (len(U_SITES) + 1)
bars = [(rf'$\mathbf{{u}}_{j}$ @ $n_{j}$', lambda c, _j=j: UC[f'iia_{c}_u{_j}'], VCOL[j])
        for j in U_SITES]
bars.append((r'$\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3$ span @ $n_3$',
             lambda c: SPACE[f'sp_iia_{c}_span_preMLP'], COL['ceil']))
for bi, (lab, get, col) in enumerate(bars):
    vals = [get(c) for c in CASE_LIST]
    ax.bar(xs + (bi - (len(bars) - 1) / 2) * W, [v.mean() for v in vals], W,
           yerr=[v.std() / np.sqrt(len(v)) for v in vals], capsize=4, color=col, alpha=0.92,
           label=lab)
ax.set_xticks(xs)
ax.set_xticklabels([f'{Y[int(c[1])]} perturbed' for c in CASE_LIST], fontsize=LABEL_FS - 1)
ax.set_ylim(0, 1.15)
style_axes(ax, ylabel='IIA')
ax.legend(fontsize=LEGEND_FS)
ax.set_title('each $\\mathbf{u}$ is causal only in its own case', fontsize=TITLE_FS - 1)
plt.tight_layout(); savefig(fig, 'u_v_cosines'); plt.show()

d_model5 = len(VDIR[1])
ang5 = lambda i, j: np.degrees(np.arccos(np.clip(C5[i, j], -1, 1)))
for i in range(5):
    for j in range(i):
        mark = '  <-- same number' if (i, j) in [(2, 0), (3, 1)] else ''
        print(f'{PLAIN5[i]} vs {PLAIN5[j]}: cos {C5[i, j]:+.3f}  ({ang5(i, j):5.1f} deg){mark}')
print(f'chance |cos| = 1/sqrt(d_model) = {1 / np.sqrt(d_model5):.4f}')
print('probe R2 / DAS fit quality: ' + '  '.join(
    f'u{j} |corr(log y{j})| {float(UC[f"u{j}_corr"]):.3f}' for j in U_SITES))

# 13. Layers 18–22 at the read-out token

Everything below is read at the **last** prompt token. Three value orders, one batch each:
$s_1>s_2>s_3$, $s_1>s_3>s_2$, $s_2>s_3>s_1$.

PCA of the residual leaving each of L18–L22, coloured by $\max(y_1,y_2,y_3)$; then the full-rank
residual interchange at each layer; then a per-head sweep at the layer where it jumps.

In [ ]:
from sklearn.decomposition import PCA

LATE_LAYERS = [18, 19, 20, 21, 22]
LATE_GROUPS = list(HEAD_GROUPS)                 # s1>s2>s3, s1>s3>s2, s2>s3>s1
LAST_POS = T - 1                                # the read-out token, where the answer is emitted


def _late():
    out = {'late_layers': np.array(LATE_LAYERS), 'last_pos': np.array(LAST_POS)}

    # --- clouds: the section-2 triples, every layer read in the same pass so rows stay aligned
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 11, LOW, HIGH)
    out['late_y'] = np.array(trips, dtype=np.int64)
    cl = MC.capture_resid_cloud(trips, T, [(l, LAST_POS) for l in LATE_LAYERS])
    for l in LATE_LAYERS:
        X = cl[(l, LAST_POS)].astype(np.float64)
        Xc = X - X.mean(0)
        p = PCA(n_components=3).fit(Xc)
        out[f'pca_coords_L{l}'] = p.transform(Xc).astype(np.float32)
        out[f'pca_evr_L{l}'] = p.explained_variance_ratio_.astype(np.float32)
    del cl
    torch.cuda.empty_cache()

    # --- full-rank residual interchange at the read-out token, one layer at a time so only one
    #     (n, T, d_model) clean capture is alive at a time
    for l in LATE_LAYERS:
        for g in LATE_GROUPS:
            bat = get_bat(HEAD_GROUPS[g])
            src = {l: S.clean_hidden_at(bat['ids_c'], l)}
            rec, iia = MC.eval_spec(dict(resid=[(l, 'full', None)]), bat, src, LAST_POS)
            out[f'resid_posrec_{ck(g)}_L{l}'] = rec.astype(np.float32)
            out[f'resid_iia_{ck(g)}_L{l}'] = iia.astype(np.float32)
            del src
            torch.cuda.empty_cache()
        print(f'  L{l}: ' + '   '.join(
            f'{g} IIA {out[f"resid_iia_{ck(g)}_L{l}"].mean():.3f}' for g in LATE_GROUPS),
            flush=True)
    return out


LATE = cached('late', _late)
LAST_POS = int(LATE['last_pos'])
print(f'read-out token {LAST_POS} of {T}')

In [ ]:
MAXV = LATE['late_y'].max(1)
ELEV, AZIM = 22, -58
nL = len(LATE_LAYERS)

fig = plt.figure(figsize=(4.2 * nL, 8.6))
for i, l in enumerate(LATE_LAYERS):
    pc, evr = LATE[f'pca_coords_L{l}'], LATE[f'pca_evr_L{l}']

    ax = fig.add_subplot(2, nL, i + 1, projection='3d')
    ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], c=MAXV, cmap=CMAP, s=5, alpha=0.6,
               edgecolors='none')
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_box_aspect((1, 1, 0.92))
    ax.set_xlabel('PC1', fontsize=TICK_FS, labelpad=4)
    ax.set_ylabel('PC2', fontsize=TICK_FS, labelpad=4)
    ax.set_zlabel('PC3', fontsize=TICK_FS, labelpad=2)
    ax.tick_params(labelsize=8, pad=0.5)
    ax.set_title(f'L{l}   EVR {evr[0]:.2f}/{evr[1]:.2f}/{evr[2]:.2f}', fontsize=TITLE_FS)

    ax = fig.add_subplot(2, nL, nL + i + 1)
    sc = ax.scatter(pc[:, 0], pc[:, 1], c=MAXV, cmap=CMAP, s=6, alpha=0.65, edgecolors='none')
    style_axes(ax, xlabel='PC1', ylabel='PC2' if i == 0 else None, grid=False)
    ax.grid(alpha=0.3)
    ax.set_title(f'L{l}', fontsize=TITLE_FS)

fig.subplots_adjust(right=0.93, hspace=0.25)
cax = fig.add_axes([0.945, 0.15, 0.010, 0.70])
cb = fig.colorbar(sc, cax=cax)
cb.set_label(r'$\max(y_1, y_2, y_3)$', fontsize=LABEL_FS)
cb.ax.tick_params(labelsize=TICK_FS)
cb.outline.set_visible(False)
savefig(fig, 'late_pca')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2))
for ax, (key, ylab) in zip(axes, [('iia', 'IIA'), ('posrec', 'position recovery')]):
    for g in LATE_GROUPS:
        v = [LATE[f'resid_{key}_{ck(g)}_L{l}'] for l in LATE_LAYERS]
        m = np.array([x.mean() for x in v])
        e = np.array([x.std() / np.sqrt(len(x)) for x in v])
        ax.errorbar(LATE_LAYERS, m, yerr=e, marker='o', ms=5.5, lw=1.8, capsize=3,
                    color=GROUP_COL[g], label=order_label(HEAD_GROUPS[g]))
    style_axes(ax, xlabel='layer patched (full residual @ read-out token)', ylabel=ylab)
    ax.set_xticks(LATE_LAYERS)
    ax.axhline(1.0, color=COL['ceil'], ls=':', lw=1.1)
    ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, 'late_layer_sweep')
plt.show()

MEAN_IIA = np.array([np.mean([LATE[f'resid_iia_{ck(g)}_L{l}'].mean() for g in LATE_GROUPS])
                     for l in LATE_LAYERS])
STEP = np.diff(MEAN_IIA)
print('mean IIA over the three orders:')
print('  ' + '   '.join(f'L{l} {m:.3f}' for l, m in zip(LATE_LAYERS, MEAN_IIA)))
print('step into each layer:')
print('  ' + '   '.join(f'L{l} {d:+.3f}' for l, d in zip(LATE_LAYERS[1:], STEP)))
JUMP_LAYER = int(LATE_LAYERS[int(np.argmax(STEP)) + 1])
print(f'largest increase: into L{JUMP_LAYER}')

## Per-head sweep at the jump layer

Each head's own output contribution at the read-out token interchanged on its own. Dashed line is
the whole attention block's output at the same layer and token.

In [ ]:
HEAD_LAYER = JUMP_LAYER          # set an int here to override the automatic pick
ATTN_SITE = ('attn', HEAD_LAYER)


def _latehead():
    out = {'head_layer': np.array(HEAD_LAYER)}
    for g in LATE_GROUPS:
        bat = get_bat(HEAD_GROUPS[g])
        ctx = MC.capture_ctx_at(bat['ids_c'], LAST_POS, HEAD_LAYER)
        att = C.capture_sites([ATTN_SITE], bat['ids_c'], LAST_POS)[ATTN_SITE]
        n_h = D.d_model // D.head_dim
        pr = np.zeros((n_h, bat['n']), dtype=np.float32)
        ii = np.zeros((n_h, bat['n']), dtype=np.float32)
        for h in range(n_h):
            pr[h], ii[h] = MC.eval_spec(
                None, bat, {}, LAST_POS,
                extra=lambda lo, hi, _h=h: MC.head_patch_handles_at(
                    HEAD_LAYER, _h, LAST_POS,
                    MC.head_out_at(ctx[lo:hi], _h, HEAD_LAYER)))
        out[f'posrec_{ck(g)}'] = pr
        out[f'iia_{ck(g)}'] = ii
        r, i = MC.eval_spec(
            None, bat, {}, LAST_POS,
            extra=lambda lo, hi: [C.block_of(ATTN_SITE).register_forward_hook(
                C.make_out_patch_hook(att[lo:hi], LAST_POS))])
        out[f'attnfull_posrec_{ck(g)}'] = r.astype(np.float32)
        out[f'attnfull_iia_{ck(g)}'] = i.astype(np.float32)
        top = np.argsort(-ii.mean(1))[:5]
        print(f'{order_label(HEAD_GROUPS[g]):>12}  attn-full IIA {i.mean():.3f}   top heads: '
              + ', '.join(f'H{t} {ii[t].mean():.3f}' for t in top), flush=True)
        del ctx, att
        torch.cuda.empty_cache()
    return out


LATEHEAD = cached('latehead', _latehead)
assert int(LATEHEAD['head_layer']) == HEAD_LAYER, \
    f"cached sweep is layer {int(LATEHEAD['head_layer'])}, not {HEAD_LAYER}; delete it or match"

In [ ]:
LATE_RANK_GROUP = 's2>s3'      # which order the x-axis is sorted by
LATE_SHOW_HEADS = None         # e.g. 16 to crop to the top ones; None = all

ordix = np.argsort(-LATEHEAD[f'posrec_{ck(LATE_RANK_GROUP)}'].mean(1))
if LATE_SHOW_HEADS:
    ordix = ordix[:LATE_SHOW_HEADS]
x, w = np.arange(len(ordix)), 0.8 / len(LATE_GROUPS)

fig, ax = plt.subplots(figsize=(15, 4.5))
for i, g in enumerate(LATE_GROUPS):
    max_slot = HEAD_GROUPS[g][0] + 1
    col = NCOL[max_slot]
    v = LATEHEAD[f'posrec_{ck(g)}'][ordix]
    ax.bar(x + (i - (len(LATE_GROUPS) - 1) / 2) * w, v.mean(1), w,
           yerr=v.std(1) / np.sqrt(v.shape[1]), color=col,
           error_kw=dict(lw=0.7, ecolor='0.3'), alpha=0.92,
           label=f'{order_label(HEAD_GROUPS[g])}  (max s{max_slot})')
    ax.axhline(LATEHEAD[f'attnfull_posrec_{ck(g)}'].mean(), ls='--', lw=1.1,
               color=col, alpha=0.8)
ax.axhline(0, color='0.5', lw=0.8)
style_axes(ax, ylabel='position recovery')
ax.set_title(f'L{HEAD_LAYER} heads, own output interchanged at the read-out token',
             fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False, ncol=3)
ax.set_xticks(x)
ax.set_xticklabels([f'H{h}' for h in ordix], rotation=90, fontsize=8)
ax.set_xlabel(f'layer {HEAD_LAYER} head', fontsize=LABEL_FS)
fig.tight_layout()
savefig(fig, f'late_heads_L{HEAD_LAYER}')
plt.show()

## Individual heads at L20

Same per-head sweep as above, layer fixed at 20 instead of the auto-picked jump layer.

In [ ]:
HEAD_LAYER_L20 = 20
ATTN_SITE_L20 = ('attn', HEAD_LAYER_L20)
LATEHEAD20_PATH = f'{RESULTS_DIR}/multicompare_latehead20_{REGIME}.npz'


def _latehead20():
    out = {'head_layer': np.array(HEAD_LAYER_L20)}
    for g in LATE_GROUPS:
        bat = get_bat(HEAD_GROUPS[g])
        ctx = MC.capture_ctx_at(bat['ids_c'], LAST_POS, HEAD_LAYER_L20)
        att = C.capture_sites([ATTN_SITE_L20], bat['ids_c'], LAST_POS)[ATTN_SITE_L20]
        n_h = D.d_model // D.head_dim
        pr = np.zeros((n_h, bat['n']), dtype=np.float32)
        ii = np.zeros((n_h, bat['n']), dtype=np.float32)
        for h in range(n_h):
            pr[h], ii[h] = MC.eval_spec(
                None, bat, {}, LAST_POS,
                extra=lambda lo, hi, _h=h: MC.head_patch_handles_at(
                    HEAD_LAYER_L20, _h, LAST_POS,
                    MC.head_out_at(ctx[lo:hi], _h, HEAD_LAYER_L20)))
        out[f'posrec_{ck(g)}'] = pr
        out[f'iia_{ck(g)}'] = ii
        r, i = MC.eval_spec(
            None, bat, {}, LAST_POS,
            extra=lambda lo, hi: [C.block_of(ATTN_SITE_L20).register_forward_hook(
                C.make_out_patch_hook(att[lo:hi], LAST_POS))])
        out[f'attnfull_posrec_{ck(g)}'] = r.astype(np.float32)
        out[f'attnfull_iia_{ck(g)}'] = i.astype(np.float32)
        top = np.argsort(-ii.mean(1))[:5]
        print(f'{order_label(HEAD_GROUPS[g]):>12}  attn-full IIA {i.mean():.3f}   top heads: '
              + ', '.join(f'H{t} {ii[t].mean():.3f}' for t in top), flush=True)
        del ctx, att
        torch.cuda.empty_cache()
    return out


if os.path.exists(LATEHEAD20_PATH):
    LATEHEAD20 = dict(np.load(LATEHEAD20_PATH, allow_pickle=True))
    print(f'loaded {LATEHEAD20_PATH}')
else:
    ensure_model()
    LATEHEAD20 = _latehead20()
    np.savez_compressed(LATEHEAD20_PATH, **{k: np.asarray(v) for k, v in LATEHEAD20.items()})
    print(f'computed and saved {LATEHEAD20_PATH}')

In [ ]:
ordix20 = np.argsort(-LATEHEAD20[f'posrec_{ck(LATE_RANK_GROUP)}'].mean(1))
if LATE_SHOW_HEADS:
    ordix20 = ordix20[:LATE_SHOW_HEADS]
x20, w20 = np.arange(len(ordix20)), 0.8 / len(LATE_GROUPS)

fig, ax = plt.subplots(figsize=(15, 4.5))
for i, g in enumerate(LATE_GROUPS):
    max_slot = HEAD_GROUPS[g][0] + 1
    col = NCOL[max_slot]
    v = LATEHEAD20[f'posrec_{ck(g)}'][ordix20]
    ax.bar(x20 + (i - (len(LATE_GROUPS) - 1) / 2) * w20, v.mean(1), w20,
           yerr=v.std(1) / np.sqrt(v.shape[1]), color=col,
           error_kw=dict(lw=0.7, ecolor='0.3'), alpha=0.92,
           label=f'{order_label(HEAD_GROUPS[g])}  (max s{max_slot})')
    ax.axhline(LATEHEAD20[f'attnfull_posrec_{ck(g)}'].mean(), ls='--', lw=1.1,
               color=col, alpha=0.8)
ax.axhline(0, color='0.5', lw=0.8)
style_axes(ax, ylabel='position recovery')
ax.set_title(f'L{HEAD_LAYER_L20} heads, own output interchanged at the read-out token',
             fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False, ncol=3)
ax.set_xticks(x20)
ax.set_xticklabels([f'H{h}' for h in ordix20], rotation=90, fontsize=8)
ax.set_xlabel(f'layer {HEAD_LAYER_L20} head', fontsize=LABEL_FS)
fig.tight_layout()
savefig(fig, f'late_heads_L{HEAD_LAYER_L20}')
plt.show()

## H27 (L20): own output and attention

Same own-output-contribution / attention-row probe as H14 in `data_sharedrep.py`, generalized to layer 20 head 27. PCA at rank 2.

In [ ]:
PROBE_LAYER, PROBE_HEAD = HEAD_LAYER_L20, 27
H27L20_PATH = f'{RESULTS_DIR}/multicompare_h27l20_{REGIME}.npz'


def _h27l20():
    out = {'probe_layer': np.array(PROBE_LAYER), 'probe_head': np.array(PROBE_HEAD)}
    for g in LATE_GROUPS:
        bat = get_bat(HEAD_GROUPS[g])
        ids = bat['ids_c']
        ctx = MC.capture_ctx_at(ids, LAST_POS, PROBE_LAYER)
        out[f'headout_{ck(g)}'] = MC.head_out_at(ctx, PROBE_HEAD, PROBE_LAYER).cpu().numpy().astype(np.float32)
        attn = np.zeros((bat['n'], T), dtype=np.float32)
        for lo in range(0, bat['n'], EVAL_BS):
            attn[lo:lo + EVAL_BS] = S.capture_attn(
                ids[lo:lo + EVAL_BS], LAST_POS, layer=PROBE_LAYER, head=PROBE_HEAD).cpu().numpy()
        out[f'attn_{ck(g)}'] = attn
        del ctx
        torch.cuda.empty_cache()
    return out


if os.path.exists(H27L20_PATH):
    H27L20 = dict(np.load(H27L20_PATH, allow_pickle=True))
    print(f'loaded {H27L20_PATH}')
else:
    ensure_model()
    H27L20 = _h27l20()
    np.savez_compressed(H27L20_PATH, **{k: np.asarray(v) for k, v in H27L20.items()})
    print(f'computed and saved {H27L20_PATH}')

In [ ]:
ALL_HEADOUT = np.concatenate([H27L20[f'headout_{ck(g)}'] for g in LATE_GROUPS], axis=0)
pca27 = PCA(n_components=2).fit(ALL_HEADOUT)
pc27 = pca27.transform(ALL_HEADOUT)

fig, ax = plt.subplots(figsize=(6, 6))
i0 = 0
for g in LATE_GROUPS:
    n = H27L20[f'headout_{ck(g)}'].shape[0]
    max_slot = HEAD_GROUPS[g][0] + 1
    ax.scatter(pc27[i0:i0 + n, 0], pc27[i0:i0 + n, 1], s=14, alpha=0.6,
               color=NCOL[max_slot], label=f'{order_label(HEAD_GROUPS[g])}  (max s{max_slot})')
    i0 += n
style_axes(ax, xlabel=f'PC1 ({pca27.explained_variance_ratio_[0]:.0%})',
           ylabel=f'PC2 ({pca27.explained_variance_ratio_[1]:.0%})')
ax.set_title(f'H{PROBE_HEAD} own output @ read-out token, L{PROBE_LAYER}', fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, f'h{PROBE_HEAD}_L{PROBE_LAYER}_pca')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(T)
for g in LATE_GROUPS:
    max_slot = HEAD_GROUPS[g][0] + 1
    a = H27L20[f'attn_{ck(g)}']
    m, se = a.mean(0), a.std(0) / np.sqrt(a.shape[0])
    ax.plot(xs, m, color=NCOL[max_slot], lw=1.8,
            label=f'{order_label(HEAD_GROUPS[g])}  (max s{max_slot})')
    ax.fill_between(xs, m - se, m + se, color=NCOL[max_slot], alpha=0.2)
for p in POS:
    ax.axvline(p, color='0.6', ls=':', lw=1)
ax.axvline(LAST_POS, color='0.3', ls='--', lw=1.2)
style_axes(ax, ylabel='attention weight')
ax.set_xticks(POS)
ax.set_xticklabels([Y[1], Y[2], Y[3]], fontsize=TICK_FS)
ax.set_title(f'H{PROBE_HEAD} attention from the read-out token, L{PROBE_LAYER}', fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, f'h{PROBE_HEAD}_L{PROBE_LAYER}_attn')
plt.show()

## H27 (L20): general input cloud

Same probe over a general cloud (no fixed order), coloured by whichever of y1/y2/y3 is largest in each sample.

In [ ]:
H27CLOUD_PATH = f'{RESULTS_DIR}/multicompare_h27l20cloud_{REGIME}.npz'


def _h27cloud():
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 29, LOW, HIGH)
    ids = D.tokenizer([MC.prompt_gen(list(t)) for t in trips],
                      return_tensors='pt')['input_ids'].to(device)
    assert ids.shape[1] == T, 'cloud prompt length must match the template'
    ctx = MC.capture_ctx_at(ids, LAST_POS, PROBE_LAYER)
    headout = MC.head_out_at(ctx, PROBE_HEAD, PROBE_LAYER).cpu().numpy().astype(np.float32)
    del ctx
    attn = np.zeros((len(trips), T), dtype=np.float32)
    for lo in range(0, len(trips), EVAL_BS):
        attn[lo:lo + EVAL_BS] = S.capture_attn(
            ids[lo:lo + EVAL_BS], LAST_POS, layer=PROBE_LAYER, head=PROBE_HEAD).cpu().numpy()
    torch.cuda.empty_cache()
    return {'y': np.array(trips, dtype=np.int64), 'headout': headout, 'attn': attn}


if os.path.exists(H27CLOUD_PATH):
    H27CLOUD = dict(np.load(H27CLOUD_PATH, allow_pickle=True))
    print(f'loaded {H27CLOUD_PATH}')
else:
    ensure_model()
    H27CLOUD = _h27cloud()
    np.savez_compressed(H27CLOUD_PATH, **{k: np.asarray(v) for k, v in H27CLOUD.items()})
    print(f'computed and saved {H27CLOUD_PATH}')

MAXSLOT27 = H27CLOUD['y'].argmax(1) + 1   # 1/2/3, whichever of y1/y2/y3 is largest in that sample

In [ ]:
pca27c = PCA(n_components=2).fit(H27CLOUD['headout'])
pc27c = pca27c.transform(H27CLOUD['headout'])

fig, ax = plt.subplots(figsize=(6, 6))
colors = [NCOL[s] for s in MAXSLOT27]
ax.scatter(pc27c[:, 0], pc27c[:, 1], s=10, alpha=0.5, color=colors, edgecolors='none')
for slot in (1, 2, 3):
    ax.scatter([], [], color=NCOL[slot], label=f'max s{slot}')
style_axes(ax, xlabel=f'PC1 ({pca27c.explained_variance_ratio_[0]:.0%})',
           ylabel=f'PC2 ({pca27c.explained_variance_ratio_[1]:.0%})')
ax.set_title(f'H{PROBE_HEAD} own output @ read-out token, L{PROBE_LAYER} (general cloud)',
             fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, f'h{PROBE_HEAD}_L{PROBE_LAYER}_pca_cloud')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(T)
for slot in (1, 2, 3):
    a = H27CLOUD['attn'][MAXSLOT27 == slot]
    m, se = a.mean(0), a.std(0) / np.sqrt(a.shape[0])
    ax.plot(xs, m, color=NCOL[slot], lw=1.8, label=f'max s{slot}  (n={a.shape[0]})')
    ax.fill_between(xs, m - se, m + se, color=NCOL[slot], alpha=0.2)
for p in POS:
    ax.axvline(p, color='0.6', ls=':', lw=1)
ax.axvline(LAST_POS, color='0.3', ls='--', lw=1.2)
style_axes(ax, ylabel='attention weight')
ax.set_xticks(POS)
ax.set_xticklabels([Y[1], Y[2], Y[3]], fontsize=TICK_FS)
ax.set_title(f'H{PROBE_HEAD} attention from the read-out token, L{PROBE_LAYER} (general cloud)',
             fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, f'h{PROBE_HEAD}_L{PROBE_LAYER}_attn_cloud')
plt.show()

## Do the n2/n3 running-max directions drive H27 and the L20 clusters?

Two rank-1 DAS directions on the L15 residual, same site + method as `DAS15` ("the comparison outcome"): n3 reuses `DAS15['basis_y3_k1']` directly (identical fit already cached); n2 is fit fresh here, on the s1>s2 flip. Each is sign-flipped in place on a general cloud; effect read at H27 (L20) and in a fresh L20-residual PCA.

In [ ]:
RUNMAX_LAYER = DAS15_LAYER   # L15, same site + method as the "comparison outcome" DAS15 fit
RUNMAX_DAS_PATH = f'{RESULTS_DIR}/multicompare_runmaxdas_{REGIME}.npz'


def _runmax_das():
    tbat = MC.make_batch(TRAIN_Q, HEAD_GROUPS['s1>s2'], T)
    ch = S.clean_hidden_at(tbat['ids_c'], RUNMAX_LAYER)
    Q, curve = MC.train_resid_das(RUNMAX_LAYER, tbat, ch, POS[1], k=1, seed=0)
    print(f'  n2: loss {curve[0]:.3f} -> {curve[-1]:.3f}', flush=True)
    torch.cuda.empty_cache()
    return {'Q_n2': Q.cpu().numpy().astype(np.float32), 'curve_n2': curve}


if os.path.exists(RUNMAX_DAS_PATH):
    RUNMAXDAS = dict(np.load(RUNMAX_DAS_PATH, allow_pickle=True))
    print(f'loaded {RUNMAX_DAS_PATH}')
else:
    ensure_model()
    RUNMAXDAS = _runmax_das()
    np.savez_compressed(RUNMAX_DAS_PATH, **{k: np.asarray(v) for k, v in RUNMAXDAS.items()})
    print(f'computed and saved {RUNMAX_DAS_PATH}')

Q_n3_runmax = DAS15['basis_y3_k1']   # already fit: L15, case y3, k=1 -- the n3 analogue

In [ ]:
RUNMAX_PROBE_PATH = f'{RESULTS_DIR}/multicompare_runmaxprobe_{REGIME}.npz'


def sign_flip_hook(Q, pos):
    """h' = h - 2 (h @ Q) Q^T at `pos` -- flips the sign of the component along the 1-D
    subspace Q, leaving its orthogonal complement untouched. A self-referential, parameter-free
    interchange: no paired counterfactual example is needed."""
    def _hook(mod, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        h = hs[:, pos, :].float()
        hs_new = hs.clone()
        hs_new[:, pos, :] = (h - 2.0 * (h @ Q) @ Q.T).to(hs.dtype)
        return (hs_new,) + out[1:] if isinstance(out, tuple) else hs_new
    return _hook


def _capture(ids, flip=None):
    """(ctx, resid20) at the read-out token, batched at EVAL_BS. `flip`: None, or (Q, pos) to
    sign-flip that residual component before the rest of the forward runs."""
    ctx_chunks, resid_chunks, store = [], [], {}

    def ctx_pre(mod, inp):
        store['ctx'] = inp[0][:, LAST_POS, :].detach().float()

    def resid_hook(mod, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        store['resid'] = hs[:, LAST_POS, :].detach().float()

    for lo in range(0, ids.shape[0], EVAL_BS):
        handles = [D.model.model.layers[PROBE_LAYER].self_attn.o_proj.register_forward_pre_hook(ctx_pre),
                   D.model.model.layers[PROBE_LAYER].register_forward_hook(resid_hook)]
        if flip is not None:
            Q, pos = flip
            handles.append(D.model.model.layers[RUNMAX_LAYER].register_forward_hook(sign_flip_hook(Q, pos)))
        try:
            with torch.no_grad():
                _ = D.model(ids[lo:lo + EVAL_BS])
        finally:
            for h in handles:
                h.remove()
        ctx_chunks.append(store.pop('ctx'))
        resid_chunks.append(store.pop('resid'))
    return torch.cat(ctx_chunks), torch.cat(resid_chunks)


def _capture_attn(ids, flip=None):
    """(n, T) attention row of PROBE_LAYER.PROBE_HEAD at LAST_POS, its own forward pass (eager
    attention) so it never shares a graph with `_capture`'s sdpa path -- same separation
    `data_sharedrep.capture_attn` uses."""
    attn = np.zeros((ids.shape[0], T), dtype=np.float32)
    for lo in range(0, ids.shape[0], EVAL_BS):
        handles = []
        if flip is not None:
            Q, pos = flip
            handles.append(D.model.model.layers[RUNMAX_LAYER].register_forward_hook(sign_flip_hook(Q, pos)))
        try:
            with torch.no_grad():
                att = D.model(ids[lo:lo + EVAL_BS], output_attentions=True).attentions[PROBE_LAYER]
        finally:
            for h in handles:
                h.remove()
        attn[lo:lo + EVAL_BS] = att[:, PROBE_HEAD, LAST_POS, :].detach().float().cpu().numpy()
    return attn


def _runmax_probe():
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 31, LOW, HIGH)
    ids = D.tokenizer([MC.prompt_gen(list(t)) for t in trips],
                      return_tensors='pt')['input_ids'].to(device)
    assert ids.shape[1] == T, 'cloud prompt length must match the template'

    Q_n2 = torch.tensor(RUNMAXDAS['Q_n2'], device=device)
    Q_n3 = torch.tensor(Q_n3_runmax, device=device)
    conditions = {'base': None, 'flip_n2': (Q_n2, POS[1]), 'flip_n3': (Q_n3, POS[2])}

    out = {'y': np.array(trips, dtype=np.int64)}
    for name, flip in conditions.items():
        ctx, resid20 = _capture(ids, flip)
        out[f'headout_{name}'] = MC.head_out_at(ctx, PROBE_HEAD, PROBE_LAYER).cpu().numpy().astype(np.float32)
        out[f'resid20_{name}'] = resid20.cpu().numpy().astype(np.float32)
        out[f'attn_{name}'] = _capture_attn(ids, flip)
        print(f'  {name}: headout mean-norm '
              f'{np.linalg.norm(out[f"headout_{name}"], axis=1).mean():.3f}', flush=True)
        del ctx, resid20
        torch.cuda.empty_cache()
    return out


if os.path.exists(RUNMAX_PROBE_PATH):
    RUNMAXPROBE = dict(np.load(RUNMAX_PROBE_PATH, allow_pickle=True))
    print(f'loaded {RUNMAX_PROBE_PATH}')
else:
    ensure_model()
    RUNMAXPROBE = _runmax_probe()
    np.savez_compressed(RUNMAX_PROBE_PATH, **{k: np.asarray(v) for k, v in RUNMAXPROBE.items()})
    print(f'computed and saved {RUNMAX_PROBE_PATH}')

MAXSLOT_RM = RUNMAXPROBE['y'].argmax(1) + 1

In [ ]:
SPACES = [('headout', f'H{PROBE_HEAD} own output'), ('resid20', f'L{PROBE_LAYER} residual')]
CONDS = [('base', 'baseline'), ('flip_n2', 'n2 direction flipped'), ('flip_n3', 'n3 direction flipped')]
colors_rm = [NCOL[s] for s in MAXSLOT_RM]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row, (space, space_lab) in enumerate(SPACES):
    pca_rm = PCA(n_components=2).fit(RUNMAXPROBE[f'{space}_base'])
    for col, (cond, cond_lab) in enumerate(CONDS):
        ax = axes[row, col]
        pc = pca_rm.transform(RUNMAXPROBE[f'{space}_{cond}'])
        ax.scatter(pc[:, 0], pc[:, 1], s=8, alpha=0.5, color=colors_rm, edgecolors='none')
        ax.set_title(f'{space_lab}: {cond_lab}', fontsize=TICK_FS)
        style_axes(ax)
for slot in (1, 2, 3):
    axes[0, 0].scatter([], [], color=NCOL[slot], label=f'max s{slot}')
axes[0, 0].legend(fontsize=LEGEND_FS, frameon=False)
fig.suptitle('Sign-flipping the running-max direction: does the cluster move?', fontsize=TITLE_FS)
fig.tight_layout()
savefig(fig, 'runmax_pca_panel')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
xs = np.arange(T)
RM_LABELS = {'base': ('0.3', 'baseline'), 'flip_n2': (NCOL[1], 'n2 flipped'),
             'flip_n3': (NCOL[2], 'n3 flipped')}
for cond, (col, lab) in RM_LABELS.items():
    a = RUNMAXPROBE[f'attn_{cond}']
    m, se = a.mean(0), a.std(0) / np.sqrt(a.shape[0])
    ax.plot(xs, m, color=col, lw=1.8, label=lab)
    ax.fill_between(xs, m - se, m + se, color=col, alpha=0.15)
for p in POS:
    ax.axvline(p, color='0.6', ls=':', lw=1)
ax.axvline(LAST_POS, color='0.3', ls='--', lw=1.2)
style_axes(ax, ylabel='attention weight')
ax.set_xticks(POS)
ax.set_xticklabels([Y[1], Y[2], Y[3]], fontsize=TICK_FS)
ax.set_title(f'H{PROBE_HEAD} attention, baseline vs. running-max sign-flip', fontsize=TITLE_FS)
ax.legend(fontsize=LEGEND_FS, frameon=False)
fig.tight_layout()
savefig(fig, 'runmax_attn_shift')
plt.show()

## Same probe, with the validated causal directions

n2: `basis_n2_larger_k1` from `explore_comparatorfig.ipynb`'s own DAS15 fit (`results/comparator_das15_2digit.npz`) -- the two-number task's own n2, already shown causal there. n3: this notebook's `DAS15['basis_y3_k1']`, same as before. Both are vectors in the model's L15 residual space, so they transfer regardless of each notebook's own prompt length; only the token position where each is applied is local to this notebook.

In [ ]:
EXT_DAS15_PATH = f'{RESULTS_DIR}/comparator_das15_{REGIME}.npz'
RUNMAX2_PROBE_PATH = f'{RESULTS_DIR}/multicompare_runmax2probe_{REGIME}.npz'

_ext_das15 = dict(np.load(EXT_DAS15_PATH, allow_pickle=True))
assert _ext_das15['basis_n2_larger_k1'].shape == (D.d_model, 1)
Q_MAP = {'y2': torch.tensor(_ext_das15['basis_n2_larger_k1'], device=device),   # comparatorfig, L15 @ y2
         'y3': torch.tensor(Q_n3_runmax, device=device)}                       # this notebook, L15 @ y3
PROBE_CASES = {'y2': (HEAD_GROUPS['s1>s2'], POS[1]),   # clean: y1>y2>y3, corrupted: win moves to y2
               'y3': (CASES['y3'], POS[2])}            # clean: y3>y2>y1, corrupted: win moves to y2


def _capture_l20(ids, patch=None):
    # L20 residual at the read-out token, in minibatches. `patch`: None, or (layer, pos, Q, ch)
    # to interchange that 1-D subspace toward ch's value at pos first -- a real clean->corrupted
    # interchange (D.make_subspace_patch_hook), not a sign flip.
    chunks, store = [], {}

    def resid_hook(mod, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        store['resid'] = hs[:, LAST_POS, :].detach().float()

    for lo in range(0, ids.shape[0], EVAL_BS):
        hi = min(lo + EVAL_BS, ids.shape[0])
        handles = [D.model.model.layers[PROBE_LAYER].register_forward_hook(resid_hook)]
        if patch is not None:
            layer, pos, Q, ch = patch
            handles.append(D.model.model.layers[layer].register_forward_hook(
                D.make_subspace_patch_hook(Q, ch[lo:hi], pos)))
        try:
            with torch.no_grad():
                _ = D.model(ids[lo:hi])
        finally:
            for h in handles:
                h.remove()
        chunks.append(store.pop('resid'))
    return torch.cat(chunks)


def _runmax2_probe():
    out = {}
    for case_name, (order, pos) in PROBE_CASES.items():
        bat = get_bat(order)
        ch = S.clean_hidden_at(bat['ids_c'], DAS15_LAYER)
        Q = Q_MAP[case_name]
        resid_clean = _capture_l20(bat['ids_c'])
        resid_corr = _capture_l20(bat['ids_r'])
        resid_patched = _capture_l20(bat['ids_r'], (DAS15_LAYER, pos, Q, ch))
        out[f'resid20_clean_{case_name}'] = resid_clean.cpu().numpy().astype(np.float32)
        out[f'resid20_corr_{case_name}'] = resid_corr.cpu().numpy().astype(np.float32)
        out[f'resid20_patched_{case_name}'] = resid_patched.cpu().numpy().astype(np.float32)
        print(f'  {case_name}: clean/corr/patched L20 norms '
              f'{resid_clean.norm(dim=-1).mean():.1f} / {resid_corr.norm(dim=-1).mean():.1f} / '
              f'{resid_patched.norm(dim=-1).mean():.1f}', flush=True)
        del ch
        torch.cuda.empty_cache()
    return out


if os.path.exists(RUNMAX2_PROBE_PATH):
    RUNMAX2PROBE = dict(np.load(RUNMAX2_PROBE_PATH, allow_pickle=True))
    print(f'loaded {RUNMAX2_PROBE_PATH}')
else:
    ensure_model()
    RUNMAX2PROBE = _runmax2_probe()
    np.savez_compressed(RUNMAX2_PROBE_PATH, **{k: np.asarray(v) for k, v in RUNMAX2PROBE.items()})
    print(f'computed and saved {RUNMAX2_PROBE_PATH}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
GROUPS_RM2 = [('clean', 'clean', COL['full']), ('corr', 'corrupted', COL['ceil']),
             ('patched', 'patched', COL['das'])]
ORDER_LAB = {'y2': ('clean: $y_1{>}y_2{>}y_3$', 'corrupted: $y_2{>}y_3{>}y_1$'),
            'y3': ('clean: $y_3{>}y_2{>}y_1$', 'corrupted: $y_2{>}y_1{>}y_3$')}

# one shared PCA basis for all three panels, fit on the natural clusters ONLY (panel 1's own
# clean/corrupted points) -- the patched points are then projected into that frame rather than
# folded into the fit, so "did patching move here" is judged against an unbiased reference.
NATURAL = [('y2', 'clean', 1), ('y2', 'corr', 2), ('y3', 'clean', 3)]   # (case, group, max slot)
pooled_all = np.concatenate([RUNMAX2PROBE[f'resid20_{g}_{c_}'] for c_, g, _ in NATURAL])
pca_shared = PCA(n_components=2).fit(pooled_all)


def proj(case_name, group):
    return pca_shared.transform(RUNMAX2PROBE[f'resid20_{group}_{case_name}'])


# --- left: the three natural clusters, coloured by which number is actually the max ---
ax = axes[0]
for c_, g, slot in NATURAL:
    pc = proj(c_, g)
    ax.scatter(pc[:, 0], pc[:, 1], s=10, alpha=0.6, color=NCOL[slot], edgecolors='none',
              label=f'max $y_{slot}$')
style_axes(ax, xlabel='PC1', ylabel='PC2')
ax.set_title('the three clusters', fontsize=TITLE_FS - 1)
ax.legend(fontsize=LEGEND_FS, frameon=False)

# --- middle / right: clean -> corrupted interchange along the validated DAS direction ---
for ax, case_name in zip(axes[1:], ['y2', 'y3']):
    for g, lab, col in GROUPS_RM2:
        pc = proj(case_name, g)
        ax.scatter(pc[:, 0], pc[:, 1], s=10, alpha=0.6, color=col, edgecolors='none', label=lab)
    style_axes(ax, xlabel='PC1', ylabel='PC2')
    ax.set_title(f'L15 DAS from ${case_name[0]}_{case_name[1]}$ patched', fontsize=TITLE_FS - 1)
    clean_lab, corr_lab = ORDER_LAB[case_name]
    ax.text(0.03, 0.03, f'{clean_lab}\n{corr_lab}', transform=ax.transAxes, fontsize=TICK_FS - 2,
            ha='left', va='bottom')
axes[1].legend(fontsize=LEGEND_FS, frameon=False)
fig.suptitle('clean -> corrupted interchange along the validated DAS direction, read at L20 '
            '(shared basis, fit on the natural clusters only)', fontsize=TITLE_FS)
fig.tight_layout()
savefig(fig, 'runmax2_pca_panel')
plt.show()

## Is the L20 2D PCA (section 13) causal?

Fits a 2-component PCA on a fresh general cloud's L20 residual, then interchanges just that 2-D subspace with the existing HEAD_GROUPS counterfactuals -- same DAS-style subspace patch as everywhere else, `P` = PCA components instead of a fitted DAS basis. Position recovery / IIA compared against the already-cached full-rank L20 ceiling (`LATE`).

In [ ]:
PCA20_PATH = f'{RESULTS_DIR}/multicompare_pca20causal_{REGIME}.npz'


def _pca20_causal():
    trips = MC.sample_cloud_triples(N_CLOUD, SEED + 41, LOW, HIGH)
    resid20 = MC.capture_resid_cloud(trips, T, [(PROBE_LAYER, LAST_POS)])[(PROBE_LAYER, LAST_POS)]
    pca20 = PCA(n_components=2).fit(resid20)
    P = torch.tensor(pca20.components_.T, device=device, dtype=torch.float32)   # (d_model, 2)

    out = {'evr': pca20.explained_variance_ratio_.astype(np.float32)}
    for g in LATE_GROUPS:
        bat = get_bat(HEAD_GROUPS[g])
        ch = S.clean_hidden_at(bat['ids_c'], PROBE_LAYER)
        rec, iia = MC.eval_spec(dict(resid=[(PROBE_LAYER, 'sub', P)]),
                                bat, {PROBE_LAYER: ch}, LAST_POS)
        out[f'posrec_{ck(g)}'] = rec.astype(np.float32)
        out[f'iia_{ck(g)}'] = iia.astype(np.float32)
        print(f'  {order_label(HEAD_GROUPS[g]):>12}  2D-PCA IIA {iia.mean():.3f}   '
              f'posrec {rec.mean():.3f}', flush=True)
        del ch
        torch.cuda.empty_cache()
    return out


if os.path.exists(PCA20_PATH):
    PCA20 = dict(np.load(PCA20_PATH, allow_pickle=True))
    print(f'loaded {PCA20_PATH}')
else:
    ensure_model()
    PCA20 = _pca20_causal()
    np.savez_compressed(PCA20_PATH, **{k: np.asarray(v) for k, v in PCA20.items()})
    print(f'computed and saved {PCA20_PATH}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
x = np.arange(len(LATE_GROUPS))
w = 0.35
for ax, (key, ylab) in zip(axes, [('iia', 'IIA'), ('posrec', 'position recovery')]):
    full = np.array([LATE[f'resid_{key}_{ck(g)}_L{PROBE_LAYER}'].mean() for g in LATE_GROUPS])
    full_se = np.array([LATE[f'resid_{key}_{ck(g)}_L{PROBE_LAYER}'].std()
                        / np.sqrt(LATE[f'resid_{key}_{ck(g)}_L{PROBE_LAYER}'].shape[0])
                        for g in LATE_GROUPS])
    pv = np.array([PCA20[f'{key}_{ck(g)}'].mean() for g in LATE_GROUPS])
    pse = np.array([PCA20[f'{key}_{ck(g)}'].std() / np.sqrt(PCA20[f'{key}_{ck(g)}'].shape[0])
                    for g in LATE_GROUPS])
    ax.bar(x - w / 2, full, w, yerr=full_se, color=COL['full'], alpha=0.9,
           error_kw=dict(lw=0.7, ecolor='0.3'), label='full-rank L20')
    ax.bar(x + w / 2, pv, w, yerr=pse, color=COL['pc1'], alpha=0.9,
           error_kw=dict(lw=0.7, ecolor='0.3'), label='2D PCA subspace')
    ax.axhline(0, color='0.5', lw=0.8)
    if key == 'iia':
        ax.axhline(1.0, color=COL['ceil'], ls=':', lw=1.1)
    ax.set_xticks(x)
    ax.set_xticklabels([order_label(HEAD_GROUPS[g]) for g in LATE_GROUPS], fontsize=TICK_FS)
    style_axes(ax, ylabel=ylab)
axes[0].legend(fontsize=LEGEND_FS, frameon=False)
fig.suptitle(f'Is the L{PROBE_LAYER} 2D PCA (EVR {PCA20["evr"].sum():.0%}) causal?', fontsize=TITLE_FS)
fig.tight_layout()
savefig(fig, f'l{PROBE_LAYER}_pca2d_causal')
plt.show()